### **Student Information**
Name (optional): 范祥彥

Student ID (optional): 114065525

GitHub ID: EvilOrbsMan

Kaggle name: FanIan

Kaggle private scoreboard snapshot: 

![pic_ranking.png](./pics/pic_ranking.png)

---

# **Instructions**

This lab has **three parts**: exercises in the Master notebook, exercises and an agent conversation in the AgentPipeline notebook, and an internal Kaggle competition documented in this Homework notebook. Together they add up to **100 points**.

**Environment recommendations to solve lab 2:**
- **Part 1 and 2 exercises:** Everything can be run with only CPU.
- **Part 3 exercises:** For the competition you will probably need GPU to train your models, so it is recommended to use Colab or Kaggle if you don't have a laptop with a dedicated GPU.

## **Part 1 (40 pts):**

1. __Main Exercises (35 pts):__ Do the **take home exercises** from Sections: `1. Data Preparation` to `8. Information Retrieval and RAG`, in the DM2026-Lab2-Master Notebook. Total: `13 exercises`. Commit your code and submit the repository link to NTU Cool.

2. **Code Comments (5 pts):** **Tidy up the code in your notebook**. 

## **Part 2 (30 pts):**

1. **Agent Prompt (20 pts):** Interact with the Agent using appropriate prompts to guide it through and complete all the required tasks.

2. **Main Exercises (10 pts):** Do the remaining **take home exercises** from DM2026-Lab2-AgentPipeline Notebook. Total: `3 exercises required`. Commit your code and submit the repository link to NTU Cool.

## **Part 3 (30 pts):**

1. **Kaggle Competition Participation (15 pts):** Submit your predictions to the [Data Mining Class Kaggle Competition](https://www.kaggle.com/t/d99def2e62194b0d8f633267c32e6d1e) by **November 22**. Check Kaggle for the exact closing time. The competition concerns emotion recognition on Twitter; participation points use the Private Leaderboard ranking:
    - If your submission meets or exceeds the **baseline score (F1 score: 0.5)**, and there are no    significant issues with your report or code, you will automatically receive a base score of 5 points of the 15 points.

        The remaining 10 points will be determined by your ranking among participants who meet or exceed the baseline:

        |Ranking  |Additional Points  |
        | --- | --- |
        |Top 10% |+10 |
        |Top 20% |+9 |
        |Top 30% |+8 |
        |Top 40% |+7 |
        |Top 50% |+6 |
        |Top 60% |+5 |
        |Top 70% |+4 |
        |Top 80% |+3 |
        |Top 90% |+2 |
        |<Top 90% |+1 |

    - If your submission does not reach the baseline score, your grade will be lower than 5. The score will range from 0 to 4 points, depending on your Private Leaderboard ranking and the relative performance of students whose submissions are also below the baseline.

2. **Competition Report (15 pts)** A report section to be filled in inside this notebook in Markdown Format, we already provided you with the template below. You need to describe your work developing the model for the competition. The report should include a section describing briefly the following elements: 
* All of your AI prompts and what you decide do use.
* Your preprocessing steps.
* The feature engineering steps.
* Explanation of your model.

[Markdown Guide - Basic Syntax](https://www.markdownguide.org/basic-syntax/)

**`Things to note for Part 3:`**

* **The code used for the competition should be in this Jupyter Notebook File** `DM2026-Lab2-Homework.ipynb`.

* **Push the code used for the competition to your repository**.

* **The code should have a clear separation for the same sections of the report, AI prompts, preprocessing, feature engineering and model explanation. Briefly comment your code for easier understanding, we provide a template at the end of this notebook.**

* Save a screenshot of your Kaggle Private Leaderboard ranking as `pics/pic_ranking.png` and keep the image link in the Student Information cell at the top of this notebook.

The Kaggle competition closes on **November 22**. The three days afterward are for completing this notebook's Markdown report, checking your files, and submitting the lab by **November 25 at 11:59 pm (Wednesday)**.

Push your work to your repository, then submit its link and a backup ZIP on the corresponding NTU COOL assignment. The ZIP should contain `DM2026-Lab2-Master.ipynb`, `DM2026-Lab2-AgentPipeline.ipynb`, `DM2026-Lab2-Homework.ipynb`, both encrypted logs in `session_logs/`, and `pics/pic_ranking.png`. See `README.md` for the exact file paths and setup steps.

---

Next you will find the template report with some simple markdown syntax explanations, use it to structure your content.

You can delete the syntax suggestions after you use them.

---

# Project Report

---

## 1. Model Development

### 1.1 Preprocessing Steps
To prepare the 4,800 competition posts (3,360 training instances and 1,440 test instances across six Ekman emotion categories: `joy`: 875, `anger`: 674, `sadness`: 628, `disgust`: 454, `surprise`: 441, and `fear`: 288), we implemented a two-tier data integration and text normalization pipeline:

* **Dataset Merging & Alignment:**
  * Loaded raw text records from `final_posts.json` and merged them with `data_identification.csv` (train/test split) and `emotion.csv` (ground-truth training labels) using `post_id` as the primary key.
  * Strictly aligned the 1,440 test rows with `sampleSubmission.csv` via a left join on `post_id` to guarantee zero ordering mismatches during submission generation.
* **Baseline Lexical Normalization (`clean_post`):**
  * Stripped URLs (`http...`, `www...`), normalized user handles (`@username` $\rightarrow$ `@user`), retained hashtag words without the `#` symbol, normalized whitespace, and lowercased text for TF-IDF vectorizers while preserving emotion-carrying punctuation (`!`, `?`) and contractions.
* **Final Transformer Normalization (`light_clean`):**
  * Replaced URLs with a canonical `http` token, normalized user mentions to `@user`, and collapsed redundant whitespace, while **preserving original letter casing and punctuation**. Because cased pre-trained transformers (`RoBERTa`, `DeBERTa-v3`) rely on capitalization (e.g., all-caps shouting) and punctuation cues for emotional intensity, avoiding lowercasing retained critical signal.

---

### 1.2 Feature Engineering Steps
Short social media posts often express emotions through sarcasm, informal spelling, or paraphrasing without exact keyword overlap. Our feature representation evolved across three stages:

* **Stage 1 — Baseline Hybrid Representation (26,995 features):**
  * Combined Word TF-IDF (`ngram_range=(1, 2)`, 7,439 features), Character boundary TF-IDF (`char_wb`, `ngram_range=(3, 5)`, 19,172 features), and L2-normalized `all-MiniLM-L6-v2` embeddings (384 features) via `scipy.sparse.hstack`.
* **Stage 2 — Withdrawn External Emotion Prototype (Compliance Audit):**
  * We initially experimented with extracting emotion probabilities and embeddings from `j-hartmann/emotion-english-distilroberta-base` (Public LB `0.5982`) and `SamLowe/roberta-base-go_emotions` (Public LB `0.6370`).
  * However, a subsequent dataset audit revealed that **2,134 of 4,800 competition posts (44.5%)** appear verbatim in GoEmotions, including **519 of 1,440 test posts (36.0%)** present in GoEmotions' training split. To strictly comply with **Kaggle Foundational Rule 4b** (prohibiting indirect human annotations of test records), all externally supervised emotion models were withdrawn and deselected.
* **Stage 3 — Final 9-Family Clean Feature & Model Architecture (100% Compliant):**
  * We rebuilt a purely compliant pipeline using only competition training labels and OSI-approved (MIT / Apache 2.0) general-purpose models across three complementary paradigms:
    1. **Fine-Tuned Pre-trained Transformers (5 families):** `FacebookAI/roberta-base`, `FacebookAI/roberta-large` (2 independent seeds: Seed 1 & Seed 777), `microsoft/deberta-v3-base`, and `microsoft/deberta-v3-large`.
    2. **General-Purpose Dense Semantic Encoders (3 families):** `BAAI/bge-large-en-v1.5`, `intfloat/e5-large-v2`, and `sentence-transformers/all-mpnet-base-v2` (paired with calibrated `LinearSVC` and `LogisticRegression` heads).
    3. **Lexical Surface Representation (1 family):** Word (1–2) + Character (3–5) TF-IDF.

---

### 1.3 Explanation of Your Model
Because the dataset is imbalanced (`joy`: 875 vs. `fear`: 288) and evaluated on **Macro-$F_1$**, all models were trained and evaluated under a single shared **5-Fold Stratified Cross-Validation** split (`random_state=42`) to ensure leak-free out-of-fold (OOF) probability alignment.

* **Transformer Fine-Tuning Architecture:**
  * To stabilize fine-tuning on 3,360 short posts, each transformer backbone used **Mean + Max Pooling** concatenated over token representations, **5x Multi-Sample Dropout (MSD)**, **Layer-wise Learning Rate Decay (LLRD, decay=0.95)**, **tempered class-weighted Cross-Entropy Loss** ($\alpha = 0.45$), and **label smoothing (`0.05`)**.
* **Iterative Performance Progression (100% Compliant Clean Pipeline):**

| Stage | Model / Ensemble Configuration | 5-Fold OOF Macro-$F_1$ | Public LB Macro-$F_1$ | Status |
| :--- | :--- | :---: | :---: | :--- |
| **Baseline** | Hybrid (Word+Char TF-IDF + MiniLM) + `LinearSVC(C=0.2)` | 0.5442 | 0.5353 | Baseline |
| **Day 1** | 4-Family Clean Blend (`RoBERTa-B` 0.5946, `DeBERTa-v3-B` 0.5432, `BGE-L` 0.5623, `TF-IDF` 0.5087) | 0.6119 | 0.5912 | — |
| **Day 2** | 6-Family Clean Blend (+ `RoBERTa-L` S1 0.6015, `DeBERTa-v3-L` 0.4749) | 0.6165 | 0.6127 | — |
| **Day 3** | 7-Family Clean Balanced Arithmetic (+ `E5-Large-v2` 0.5473) | 0.6175 | 0.6189 | — |
| **Day 4** | 9-Family Arith + Full Nested-CV Multipliers (`1.0x`) | 0.6144 | 0.6085 | Ablation |
| **Day 4** | **9-Family L2 Meta-Stacker (`LogisticRegression`, `C=0.10`)** | **0.6201** | 0.6150 | **Final Selection #2** |
| **Day 4** | **9-Family Balanced Arithmetic (Raw Argmax, + `RoBERTa-L` S2 0.6100 & `MPNet` 0.4980)** | 0.6193 | **0.6202** | **Final Selection #1** |

* **Key Empirical Findings & Observations:**
  * **Numerical Stability in `DeBERTa-v3`:** Fine-tuning `DeBERTa-v3` under `FP16` mixed precision caused gradient overflow and single-class collapse (accuracy ~0.20, Macro-$F_1$ `0.0557`). Switching to pure `FP32` resolved the issue for `DeBERTa-v3-base`, while `DeBERTa-v3-large` remained sensitive and underfitted individually (`0.4749` / `0.4354`), yet still contributed positive architectural diversity to the ensemble.
  * **Overfitting of Aggressive Post-Processing:** On 3,360 samples, optimizing class multipliers via full Nested-CV (`1.0x`) degraded Public LB from `0.6202` down to `0.6085` (and Dirichlet weight search similarly overfitted OOF). Simple domain-informed arithmetic soft-voting on raw probabilities generalized best.
  * **OOF Optimism & Natural Ceiling:** Selecting the highest-F1 epoch checkpoint per fold introduces slight optimism in OOF scores. Across Day 3 and Day 4, differences between ensembling variants (`0.001`–`0.008`) corresponded to only a handful of posts, suggesting an asymptotic ceiling near `~0.62` driven by inherent label subjectivity in short texts.
  * **Final Submission Risk Hedging:** To hedge against Private Leaderboard variance, we selected two structurally distinct aggregation methods: fixed-weight arithmetic soft-voting (`sub_day4_2_9fam_arith_raw.csv`, `0.6202`) and a regularized L2 meta-stacker (`sub_day4_5_9fam_meta_l2.csv`, `0.6150`).

---

### 1.4 AI Prompts, what and why you decide to use that technique that suggest by AI
Throughout the competition, we collaborated with two AI assistants (**Gemini** and **Claude**) for pipeline engineering, debugging, and independent compliance verification:

1. **Prompt 1 — Dataset Inspection, Baseline Design & Submission Debugging (Gemini):**
   * *Prompt:* Asked how to inspect `final_posts.json`, align splits safely, build a strong CPU baseline, and later debugged Kaggle's `ID column 'id' not found` error when uploading the first CSV.
   * *AI Suggestion:* Recommended the Word+Char TF-IDF + MiniLM + Balanced `LinearSVC` hybrid pipeline and identified that Kaggle's parser required `id` instead of `post_id` as the CSV header.
   * *Decision & Rationale:* **Adopted.** Immediately resolved the submission error and established a `0.5353` Public LB baseline.
2. **Prompt 2 — Data Leakage Audit & Rule 4b Compliance Review (Claude & Gemini):**
   * *Prompt:* After noticing a sharp jump to `0.6370` using `SamLowe/roberta-base-go_emotions`, asked Claude to audit potential overlap between the competition dataset and GoEmotions and interpret Kaggle Foundational Rule 4b.
   * *AI Suggestion:* Verified a 44.5% verbatim overlap (519/1,440 test posts in GoEmotions' training set) and advised withdrawing all externally emotion-supervised models in favor of a conservative, 100% compliant pipeline.
   * *Decision & Rationale:* **Adopted (Withdrew `0.6370` prototype).** Prioritized academic integrity and strict Rule 4b compliance over an inflated Public LB score.
3. **Prompt 3 — Clean Multi-Family GPU Fine-Tuning & `DeBERTa-v3` FP32 Fix (Gemini):**
   * *Prompt:* Shared `DeBERTa-v3`'s `FP16` collapse logs and asked how to build an incremental multi-day GPU ensemble within Colab Pro and Kaggle's 5-submission daily limit.
   * *AI Suggestion:* Diagnosed `DeBERTa-v3` `FP16` overflow (switching to `FP32`), proposed Mean+Max Pooling + 5x MSD + LLRD for `RoBERTa-large`, and structured a Day 1 $\rightarrow$ Day 4 `.npz` probability handoff across 9 clean model families.
   * *Decision & Rationale:* **Adopted.** Steadily improved clean Public LB from `0.5912` $\rightarrow$ `0.6127` $\rightarrow$ `0.6189` $\rightarrow$ `0.6202`.
4. **Prompt 4 — Post-Processing Ablation & Final Submission Selection (Claude & Gemini):**
   * *Prompt:* Asked whether Nested-CV class multipliers or Dirichlet weight search could push the clean ensemble past the `~0.62` plateau, and how to select the final 2 submissions.
   * *AI Suggestion:* Tested damped (`0.5x`) and full (`1.0x`) Nested-CV multipliers; Claude further advised hedging Private LB risk by pairing two distinct aggregation methods (arithmetic soft-voting + L2 meta-stacker) rather than two near-identical blends.
   * *Decision & Rationale:* **Partially Adopted / Rejected Overfitting.** Rejected full multipliers (`0.6085`) and Dirichlet search after empirical validation showed overfitting; adopted the raw 9-Family arithmetic blend (`0.6202`) paired with the L2 meta-stacker (`0.6150`) as our final `2/2` selections.

**`From here on starts the code section for the competition.`**

---

# **Competition Code**

## 1. Preprocessing Steps

In [3]:
import json
import re
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import f1_score, classification_report

# 1. Load raw competition files from data/kaggle
kaggle_dir = "data/kaggle"
with open(f"{kaggle_dir}/final_posts.json", "r", encoding="utf-8") as f:
    posts_df = pd.DataFrame(json.load(f))

id_df = pd.read_csv(f"{kaggle_dir}/data_identification.csv")
emo_df = pd.read_csv(f"{kaggle_dir}/emotion.csv")
sub_df = pd.read_csv(f"{kaggle_dir}/sampleSubmission.csv")

# 2. Merge posts with split identification and training labels
df = posts_df.merge(id_df, on="post_id", how="left").merge(emo_df, on="post_id", how="left")

# 3. Text cleaning function preserving emotion-relevant punctuation (!, ?) and contractions
def clean_post(text: str) -> str:
    text = str(text)
    text = re.sub(r"http\S+|www\.\S+", " ", text)       # Remove URLs
    text = re.sub(r"@\w+", "@user", text)              # Normalize user mentions
    text = re.sub(r"#(\w+)", r"\1", text)              # Keep hashtag word, strip '#' symbol
    text = re.sub(r"\s+", " ", text).strip().lower()   # Normalize whitespace and lowercase
    return text

df["clean_text"] = df["text"].apply(clean_post)

# 4. Separate train (3,360 rows) and test (1,440 rows, ordered exactly like sampleSubmission.csv)
train_df = df[df["set"] == "train"].reset_index(drop=True)
test_df = sub_df[["post_id"]].merge(df[["post_id", "text", "clean_text"]], on="post_id", how="left")

print(f"Train shape: {train_df.shape}, Missing texts: {train_df['clean_text'].isna().sum()}")
print(f"Test shape : {test_df.shape}, Missing texts: {test_df['clean_text'].isna().sum()}")
print("\nSample cleaned texts:")
print(train_df[["post_id", "label", "clean_text"]].head(3))

Train shape: (3360, 5), Missing texts: 0
Test shape : (1440, 3), Missing texts: 0

Sample cleaned texts:
   post_id    label                        clean_text
0  f8bv5zh  disgust  thought it looked weird as well.
1  q0rr1l9  disgust                bear died in a pit
2  p94rwgk  sadness             i have endured worse.


## 2. Feature Engineering Steps

In [4]:
from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize
from sentence_transformers import SentenceTransformer

# 1. Word-level TF-IDF (captures unigrams and bigrams with sublinear TF scaling)
word_tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True,
)
X_train_word = word_tfidf.fit_transform(train_df["clean_text"])
X_test_word = word_tfidf.transform(test_df["clean_text"])

# 2. Character-level TF-IDF (captures subword morphology, suffixes, and punctuation emphasis)
char_tfidf = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 5),
    min_df=3,
    max_df=0.95,
    sublinear_tf=True,
)
X_train_char = char_tfidf.fit_transform(train_df["clean_text"])
X_test_char = char_tfidf.transform(test_df["clean_text"])

# 3. Pre-trained MiniLM dense semantic embeddings (384 dimensions, L2-normalized)
sbert = SentenceTransformer("all-MiniLM-L6-v2")
X_train_sbert = normalize(sbert.encode(train_df["text"].tolist(), show_progress_bar=True))
X_test_sbert = normalize(sbert.encode(test_df["text"].tolist(), show_progress_bar=True))

# 4. Combine lexical TF-IDF features and semantic embeddings into a unified feature matrix
X_train_tfidf_all = hstack([X_train_word, X_train_char])
X_test_tfidf_all = hstack([X_test_word, X_test_char])

X_train_hybrid = hstack([X_train_word, X_train_char, csr_matrix(X_train_sbert * 1.5)])
X_test_hybrid = hstack([X_test_word, X_test_char, csr_matrix(X_test_sbert * 1.5)])

y_train = train_df["label"].values

print(f"Word TF-IDF shape  : {X_train_word.shape}")
print(f"Char TF-IDF shape  : {X_train_char.shape}")
print(f"MiniLM shape       : {X_train_sbert.shape}")
print(f"Hybrid matrix shape: {X_train_hybrid.shape}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/105 [00:00<?, ?it/s]

Batches:   0%|          | 0/45 [00:00<?, ?it/s]

Word TF-IDF shape  : (3360, 7439)
Char TF-IDF shape  : (3360, 19172)
MiniLM shape       : (3360, 384)
Hybrid matrix shape: (3360, 26995)


## 3. Model Implementation Steps

In [12]:
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Evaluate candidate representations and classifiers using 5-fold Stratified CV (Macro F1)
candidates = [
    ("Word TF-IDF + LinearSVC (C=0.5)", X_train_word, X_test_word,
     LinearSVC(C=0.5, class_weight="balanced", random_state=42)),
    ("Word+Char TF-IDF + LinearSVC (C=0.3)", X_train_tfidf_all, X_test_tfidf_all,
     LinearSVC(C=0.3, class_weight="balanced", random_state=42)),
    ("MiniLM + LinearSVC (C=0.5)", X_train_sbert, X_test_sbert,
     LinearSVC(C=0.5, class_weight="balanced", random_state=42)),
    ("MiniLM + LogisticRegression (C=2.0)", X_train_sbert, X_test_sbert,
     LogisticRegression(C=2.0, class_weight="balanced", max_iter=1000, random_state=42)),
    ("Hybrid (TF-IDF + MiniLM) + LinearSVC (C=0.2)", X_train_hybrid, X_test_hybrid,
     LinearSVC(C=0.2, class_weight="balanced", random_state=42)),
    ("Hybrid (TF-IDF + MiniLM) + LogReg (C=1.5)", X_train_hybrid, X_test_hybrid,
     LogisticRegression(C=1.5, class_weight="balanced", max_iter=1000, random_state=42)),
]

best_score = -1.0
best_name = None
best_model = None
best_X_tr, best_X_te = None, None

print("=== 5-Fold Stratified Cross-Validation Results ===")
for name, X_tr, X_te, clf in candidates:
    f1_macro = cross_val_score(clf, X_tr, y_train, cv=skf, scoring="f1_macro", n_jobs=-1).mean()
    f1_weighted = cross_val_score(clf, X_tr, y_train, cv=skf, scoring="f1_weighted", n_jobs=-1).mean()
    print(f"{name:45s} | CV Macro F1: {f1_macro:.4f} | Weighted F1: {f1_weighted:.4f}")
    if f1_macro > best_score:
        best_score = f1_macro
        best_name = name
        best_model = clf
        best_X_tr, best_X_te = X_tr, X_te

print(f"\nSelected Best Model: {best_name} (CV Macro F1 = {best_score:.4f})")

# Fit the winning model on the full training set and generate predictions
best_model.fit(best_X_tr, y_train)
test_preds = best_model.predict(best_X_te)

submission = pd.DataFrame({
    "id": test_df["post_id"],
    "label": test_preds,
})
submission.to_csv("submission_baseline.csv", index=False)

print(f"\nSaved 'submission_baseline.csv' with shape {submission.shape}")
print("Predicted test label distribution:")
print(submission["label"].value_counts())
print("\nFirst 5 rows of submission_baseline.csv:")
print(submission.head())

=== 5-Fold Stratified Cross-Validation Results ===
Word TF-IDF + LinearSVC (C=0.5)               | CV Macro F1: 0.4806 | Weighted F1: 0.4981
Word+Char TF-IDF + LinearSVC (C=0.3)          | CV Macro F1: 0.5046 | Weighted F1: 0.5225
MiniLM + LinearSVC (C=0.5)                    | CV Macro F1: 0.5024 | Weighted F1: 0.5255
MiniLM + LogisticRegression (C=2.0)           | CV Macro F1: 0.4982 | Weighted F1: 0.5197
Hybrid (TF-IDF + MiniLM) + LinearSVC (C=0.2)  | CV Macro F1: 0.5442 | Weighted F1: 0.5634
Hybrid (TF-IDF + MiniLM) + LogReg (C=1.5)     | CV Macro F1: 0.5396 | Weighted F1: 0.5590

Selected Best Model: Hybrid (TF-IDF + MiniLM) + LinearSVC (C=0.2) (CV Macro F1 = 0.5442)

Saved 'submission_baseline.csv' with shape (1440, 2)
Predicted test label distribution:
label
joy         399
anger       309
sadness     258
surprise    174
disgust     167
fear        133
Name: count, dtype: int64

First 5 rows of submission_baseline.csv:
        id     label
0  xyzx9jp     anger
1  l3vzzxt      fe

In [13]:
import os, glob
import pandas as pd

print("Data files:", glob.glob("data/kaggle/*"))
for name in ["X_train_full", "X_test_full", "y_train", "train_df", "test_df"]:
    if name in globals():
        obj = globals()[name]
        print(f"{name}: {type(obj).__name__}, shape={getattr(obj, 'shape', len(obj))}")
    else:
        print(f"{name}: not in memory (kernel restarted)")

Data files: ['data/kaggle\\data_identification.csv', 'data/kaggle\\emotion.csv', 'data/kaggle\\final_posts.json', 'data/kaggle\\sampleSubmission.csv']
X_train_full: not in memory (kernel restarted)
X_test_full: not in memory (kernel restarted)
y_train: StringArray, shape=(3360,)
train_df: DataFrame, shape=(3360, 5)
test_df: DataFrame, shape=(1440, 3)


In [15]:
print("Label distribution:\n", train_df["label"].value_counts())

try:
    import transformers
    print("transformers version:", transformers.__version__)
except ImportError:
    print("transformers not installed")

Label distribution:
 label
joy         875
anger       674
sadness     628
disgust     454
surprise    441
fear        288
Name: count, dtype: int64
transformers version: 5.17.0


> **Compliance & Withdrawal Notice (Kaggle Foundational Rule 4b):**  
> The code cells below—up to the **"Final 100% Compliant 9-Family Clean Pipeline"** section—contain earlier experiments using externally supervised emotion models (`j-hartmann/emotion-english-distilroberta-base` and `SamLowe/roberta-base-go_emotions`). During a subsequent data audit, we found that 2,134 of 4,800 competition posts (44.5%) appear verbatim in GoEmotions, including 519 of 1,440 test posts (36.0%). To strictly adhere to Kaggle Foundational Rule 4b (prohibiting indirect human annotations of test records), these models were **withdrawn and deselected** from our final Kaggle submissions. All final scored predictions (`0.6202` and `0.6150`) were produced exclusively by the 100% compliant 9-family clean pipeline described below.

In [16]:
import os
import time
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import normalize
from sklearn.metrics import f1_score

torch.set_num_threads(min(16, os.cpu_count() or 8))
cache_path = "data/kaggle/cached_emotion_features.npz"

train_texts_raw = train_df["text"].fillna("").astype(str).tolist()
test_texts_raw  = test_df["text"].fillna("").astype(str).tolist()
train_texts_clean = train_df["clean_text"].fillna("").astype(str).tolist()
test_texts_clean  = test_df["clean_text"].fillna("").astype(str).tolist()
y_true = train_df["label"].astype(str).to_numpy()

if os.path.exists(cache_path):
    print(f"Loading cached features from {cache_path}...")
    data = np.load(cache_path)
    X_train_minilm = data["X_train_minilm"]
    X_test_minilm  = data["X_test_minilm"]
    X_train_emo_emb = data["X_train_emo_emb"]
    X_test_emo_emb  = data["X_test_emo_emb"]
    X_train_emo_prob = data["X_train_emo_prob"]
    X_test_emo_prob  = data["X_test_emo_prob"]
    X_train_emo_logit = data["X_train_emo_logit"]
    X_test_emo_logit  = data["X_test_emo_logit"]
else:
    t0 = time.time()
    print("1/2 Extracting all-MiniLM-L6-v2 embeddings...")
    sbert = SentenceTransformer("all-MiniLM-L6-v2", device="cpu")
    X_train_minilm = sbert.encode(train_texts_clean, batch_size=128, show_progress_bar=True, normalize_embeddings=True)
    X_test_minilm  = sbert.encode(test_texts_clean, batch_size=128, show_progress_bar=True, normalize_embeddings=True)

    print("2/2 Extracting j-hartmann/emotion-english-distilroberta-base embeddings & logits...")
    emo_model_name = "j-hartmann/emotion-english-distilroberta-base"
    tokenizer = AutoTokenizer.from_pretrained(emo_model_name)
    emo_model = AutoModelForSequenceClassification.from_pretrained(emo_model_name, output_hidden_states=True)
    emo_model.eval()

    def extract_distilroberta(texts, batch_size=64):
        all_emb, all_prob, all_logit = [], [], []
        for i in range(0, len(texts), batch_size):
            batch = texts[i : i + batch_size]
            enc = tokenizer(batch, padding=True, truncation=True, max_length=128, return_tensors="pt")
            with torch.inference_mode():
                out = emo_model(**enc)
                logits = out.logits
                probs = torch.softmax(logits, dim=-1)
                # CLS token from last hidden layer + attention-masked mean pooling
                last_hidden = out.hidden_states[-1]
                cls_emb = last_hidden[:, 0, :]
                mask = enc["attention_mask"].unsqueeze(-1).float()
                mean_emb = (last_hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)
                combined_emb = (cls_emb + mean_emb) * 0.5
            all_emb.append(combined_emb.numpy())
            all_prob.append(probs.numpy())
            all_logit.append(logits.numpy())
            if (i // batch_size + 1) % 15 == 0 or i + batch_size >= len(texts):
                print(f"  Processed {min(i + batch_size, len(texts))}/{len(texts)} posts ({time.time() - t0:.1f}s)")
        emb = normalize(np.vstack(all_emb))
        prob = np.vstack(all_prob)
        logit = np.vstack(all_logit)
        return emb.astype(np.float32), prob.astype(np.float32), logit.astype(np.float32)

    X_train_emo_emb, X_train_emo_prob, X_train_emo_logit = extract_distilroberta(train_texts_raw)
    X_test_emo_emb,  X_test_emo_prob,  X_test_emo_logit  = extract_distilroberta(test_texts_raw)

    np.savez_compressed(
        cache_path,
        X_train_minilm=X_train_minilm, X_test_minilm=X_test_minilm,
        X_train_emo_emb=X_train_emo_emb, X_test_emo_emb=X_test_emo_emb,
        X_train_emo_prob=X_train_emo_prob, X_test_emo_prob=X_test_emo_prob,
        X_train_emo_logit=X_train_emo_logit, X_test_emo_logit=X_test_emo_logit,
    )
    print(f"Saved cached features to {cache_path} in {time.time() - t0:.1f}s!")

print("Shapes:", X_train_minilm.shape, X_train_emo_emb.shape, X_train_emo_prob.shape)

1/2 Extracting all-MiniLM-L6-v2 embeddings...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/27 [00:00<?, ?it/s]

Batches:   0%|          | 0/12 [00:00<?, ?it/s]

2/2 Extracting j-hartmann/emotion-english-distilroberta-base embeddings & logits...


config.json:   0%|          | 0.00/1.00k [00:00<?, ?B/s]

... (7 outputs left out)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  329MB            

model.safetensors: downloading bytes:           |  0.00B            

  Processed 960/3360 posts (193.4s)
  Processed 1920/3360 posts (237.1s)
  Processed 2880/3360 posts (281.1s)
  Processed 3360/3360 posts (297.9s)
  Processed 960/1440 posts (332.8s)
  Processed 1440/1440 posts (355.6s)
Saved cached features to data/kaggle/cached_emotion_features.npz in 357.2s!
Shapes: (3360, 384) (3360, 768) (3360, 7)


In [17]:
import numpy as np
import pandas as pd
from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report

# 1. Build Word + Char TF-IDF matrices
word_vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=25000, sublinear_tf=True)
char_vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=3, max_features=35000, sublinear_tf=True)

X_tr_word = word_vec.fit_transform(train_texts_clean)
X_te_word = word_vec.transform(test_texts_clean)
X_tr_char = char_vec.fit_transform(train_texts_clean)
X_te_char = char_vec.transform(test_texts_clean)

# Baseline feature matrix (Word + Char TF-IDF + MiniLM)
X_tr_base = hstack([X_tr_word, X_tr_char, csr_matrix(X_train_minilm)]).tocsr()
X_te_base = hstack([X_te_word, X_te_char, csr_matrix(X_test_minilm)]).tocsr()

# Scaled emotion logits/probs for linear & neural models
scaler_emo = StandardScaler()
X_tr_emo_scaled = scaler_emo.fit_transform(np.hstack([X_train_emo_prob, X_train_emo_logit * 0.2]))
X_te_emo_scaled = scaler_emo.transform(np.hstack([X_test_emo_prob, X_test_emo_logit * 0.2]))

# Enhanced Hybrid feature matrix (Word + Char TF-IDF + MiniLM + DistilRoBERTa-Emotion Emb + Scaled Emotion Probs/Logits)
X_tr_full = hstack([
    X_tr_word,
    X_tr_char,
    csr_matrix(X_train_minilm * 1.0),
    csr_matrix(X_train_emo_emb * 1.5),
    csr_matrix(X_tr_emo_scaled * 0.35),
]).tocsr()

X_te_full = hstack([
    X_te_word,
    X_te_char,
    csr_matrix(X_test_minilm * 1.0),
    csr_matrix(X_test_emo_emb * 1.5),
    csr_matrix(X_te_emo_scaled * 0.35),
]).tocsr()

# Dense feature matrix for MLP & Logistic Regression on Embeddings
X_tr_dense = np.hstack([X_train_emo_emb * 1.5, X_train_minilm, X_tr_emo_scaled * 0.5])
X_te_dense = np.hstack([X_test_emo_emb * 1.5, X_test_minilm, X_te_emo_scaled * 0.5])

le = LabelEncoder()
y_enc = le.fit_transform(y_true)
classes = le.classes_
n_classes = len(classes)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# --- Ablation 1: Baseline (Word+Char TF-IDF + MiniLM -> LinearSVC C=0.2 balanced) ---
oof_base = np.zeros(len(y_enc), dtype=int)
for tr_idx, va_idx in skf.split(X_tr_base, y_enc):
    clf = LinearSVC(C=0.2, class_weight="balanced", random_state=42)
    clf.fit(X_tr_base[tr_idx], y_enc[tr_idx])
    oof_base[va_idx] = clf.predict(X_tr_base[va_idx])
f1_base = f1_score(y_enc, oof_base, average="macro")

# --- Ablation 2: Single LinearSVC on Full Emotion-Enhanced Features ---
oof_svc_full = np.zeros(len(y_enc), dtype=int)
for tr_idx, va_idx in skf.split(X_tr_full, y_enc):
    clf = LinearSVC(C=0.15, class_weight="balanced", random_state=42)
    clf.fit(X_tr_full[tr_idx], y_enc[tr_idx])
    oof_svc_full[va_idx] = clf.predict(X_tr_full[va_idx])
f1_svc_full = f1_score(y_enc, oof_svc_full, average="macro")

# --- Ablation 3: 5-Fold Out-of-Fold Probability Ensemble (Calibrated SVC + LogReg + Dense LogReg/MLP) ---
oof_prob_svc = np.zeros((len(y_enc), n_classes))
oof_prob_lr_full = np.zeros((len(y_enc), n_classes))
oof_prob_lr_dense = np.zeros((len(y_enc), n_classes))
oof_prob_mlp = np.zeros((len(y_enc), n_classes))

test_prob_svc = np.zeros((len(test_df), n_classes))
test_prob_lr_full = np.zeros((len(test_df), n_classes))
test_prob_lr_dense = np.zeros((len(test_df), n_classes))
test_prob_mlp = np.zeros((len(test_df), n_classes))

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_tr_full, y_enc)):
    # Model A: Calibrated LinearSVC on full sparse+dense hybrid
    base_svc = LinearSVC(C=0.15, class_weight="balanced", random_state=42 + fold)
    cal_svc = CalibratedClassifierCV(base_svc, cv=3, method="sigmoid")
    cal_svc.fit(X_tr_full[tr_idx], y_enc[tr_idx])
    oof_prob_svc[va_idx] = cal_svc.predict_proba(X_tr_full[va_idx])
    test_prob_svc += cal_svc.predict_proba(X_te_full) / skf.n_splits

    # Model B: LogisticRegression (balanced) on full hybrid
    lr_full = LogisticRegression(C=0.8, class_weight="balanced", max_iter=1000, random_state=42 + fold)
    lr_full.fit(X_tr_full[tr_idx], y_enc[tr_idx])
    oof_prob_lr_full[va_idx] = lr_full.predict_proba(X_tr_full[va_idx])
    test_prob_lr_full += lr_full.predict_proba(X_te_full) / skf.n_splits

    # Model C: LogisticRegression (natural prior) on dense emotion embeddings + probs
    lr_dense = LogisticRegression(C=0.5, max_iter=1000, random_state=42 + fold)
    lr_dense.fit(X_tr_dense[tr_idx], y_enc[tr_idx])
    oof_prob_lr_dense[va_idx] = lr_dense.predict_proba(X_tr_dense[va_idx])
    test_prob_lr_dense += lr_dense.predict_proba(X_te_dense) / skf.n_splits

    # Model D: Shallow regularized MLP on dense emotion embeddings + probs
    mlp = MLPClassifier(hidden_layer_sizes=(128,), alpha=1.5, max_iter=40, early_stopping=True, random_state=42 + fold)
    mlp.fit(X_tr_dense[tr_idx], y_enc[tr_idx])
    oof_prob_mlp[va_idx] = mlp.predict_proba(X_tr_dense[va_idx])
    test_prob_mlp += mlp.predict_proba(X_te_dense) / skf.n_splits

f1_m_svc = f1_score(y_enc, oof_prob_svc.argmax(axis=1), average="macro")
f1_m_lr_full = f1_score(y_enc, oof_prob_lr_full.argmax(axis=1), average="macro")
f1_m_lr_dense = f1_score(y_enc, oof_prob_lr_dense.argmax(axis=1), average="macro")
f1_m_mlp = f1_score(y_enc, oof_prob_mlp.argmax(axis=1), average="macro")

# Weighted soft-voting ensemble
w_svc, w_lrf, w_lrd, w_mlp = 0.35, 0.30, 0.20, 0.15
oof_ens_prob = w_svc * oof_prob_svc + w_lrf * oof_prob_lr_full + w_lrd * oof_prob_lr_dense + w_mlp * oof_prob_mlp
test_ens_prob = w_svc * test_prob_svc + w_lrf * test_prob_lr_full + w_lrd * test_prob_lr_dense + w_mlp * test_prob_mlp
f1_ens_raw = f1_score(y_enc, oof_ens_prob.argmax(axis=1), average="macro")

# --- Ablation 4: Nested Out-of-Fold Class Multiplier Tuning (prevents overfitting) ---
def fit_class_multipliers(prob_matrix, y_target, n_rounds=3):
    mult = np.ones(n_classes, dtype=np.float64)
    best_score = f1_score(y_target, prob_matrix.argmax(axis=1), average="macro")
    grid = np.linspace(0.70, 1.35, 66)
    for _ in range(n_rounds):
        for c in range(n_classes):
            best_m = mult[c]
            for m in grid:
                mult[c] = m
                preds = (prob_matrix * mult).argmax(axis=1)
                score = f1_score(y_target, preds, average="macro")
                if score > best_score + 1e-5:
                    best_score = score
                    best_m = m
            mult[c] = best_m
    # Shrink 30% toward 1.0 to regularize for unseen test data
    return 0.7 * mult + 0.3 * np.ones_like(mult)

oof_tuned_preds = np.zeros(len(y_enc), dtype=int)
fold_multipliers = []
for tr_idx, va_idx in skf.split(oof_ens_prob, y_enc):
    m_fold = fit_class_multipliers(oof_ens_prob[tr_idx], y_enc[tr_idx])
    fold_multipliers.append(m_fold)
    oof_tuned_preds[va_idx] = (oof_ens_prob[va_idx] * m_fold).argmax(axis=1)

f1_ens_tuned = f1_score(y_enc, oof_tuned_preds, average="macro")
final_multipliers = np.mean(fold_multipliers, axis=0)

print("=" * 72)
print("5-FOLD STRATIFIED CV MACRO-F1 ABLATION SUMMARY")
print("=" * 72)
print(f"1. Baseline (Word+Char TF-IDF + MiniLM -> LinearSVC C=0.2) : {f1_base:.4f}")
print(f"2. + DistilRoBERTa-Emotion Features (LinearSVC C=0.15)     : {f1_svc_full:.4f} ({f1_svc_full - f1_base:+.4f})")
print(f"   - Component A: Calibrated LinearSVC (Full Hybrid)       : {f1_m_svc:.4f}")
print(f"   - Component B: LogisticRegression Balanced (Full Hybrid): {f1_m_lr_full:.4f}")
print(f"   - Component C: LogisticRegression (Dense Emotion+MiniLM): {f1_m_lr_dense:.4f}")
print(f"   - Component D: Regularized MLP (Dense Emotion+MiniLM)   : {f1_m_mlp:.4f}")
print(f"3. Multi-Model Soft-Voting Probability Ensemble            : {f1_ens_raw:.4f} ({f1_ens_raw - f1_base:+.4f})")
print(f"4. Ensemble + Nested OOF Class-Multiplier Tuning           : {f1_ens_tuned:.4f} ({f1_ens_tuned - f1_base:+.4f})")
print("=" * 72)
print("Learned Regularized Class Multipliers:", dict(zip(classes, np.round(final_multipliers, 3))))
print("\nDetailed 5-Fold OOF Classification Report (Best Model):")
best_oof_preds = oof_tuned_preds if f1_ens_tuned >= f1_ens_raw else oof_ens_prob.argmax(axis=1)
print(classification_report(y_true, le.inverse_transform(best_oof_preds), digits=4))

# Generate submission_improved.csv using the higher CV strategy
if f1_ens_tuned >= f1_ens_raw:
    final_test_preds = le.inverse_transform((test_ens_prob * final_multipliers).argmax(axis=1))
    chosen_tag = "Ensemble + Nested OOF Multiplier Tuning"
else:
    final_test_preds = le.inverse_transform(test_ens_prob.argmax(axis=1))
    chosen_tag = "Multi-Model Soft-Voting Ensemble"

sub_improved = pd.DataFrame({
    "id": test_df["post_id"].astype(str),
    "emotion": final_test_preds
})
sub_improved.to_csv("submission_improved.csv", index=False)
print(f"\nSaved submission_improved.csv ({chosen_tag}), shape={sub_improved.shape}")
print("Predicted test distribution:\n", sub_improved["emotion"].value_counts())

5-FOLD STRATIFIED CV MACRO-F1 ABLATION SUMMARY
1. Baseline (Word+Char TF-IDF + MiniLM -> LinearSVC C=0.2) : 0.5514
2. + DistilRoBERTa-Emotion Features (LinearSVC C=0.15)     : 0.6105 (+0.0590)
   - Component A: Calibrated LinearSVC (Full Hybrid)       : 0.6212
   - Component B: LogisticRegression Balanced (Full Hybrid): 0.6090
   - Component C: LogisticRegression (Dense Emotion+MiniLM): 0.6194
   - Component D: Regularized MLP (Dense Emotion+MiniLM)   : 0.6115
3. Multi-Model Soft-Voting Probability Ensemble            : 0.6202 (+0.0688)
4. Ensemble + Nested OOF Class-Multiplier Tuning           : 0.6194 (+0.0679)
Learned Regularized Class Multipliers: {'anger': np.float64(1.112), 'disgust': np.float64(1.101), 'fear': np.float64(0.926), 'joy': np.float64(1.017), 'sadness': np.float64(0.975), 'surprise': np.float64(0.971)}

Detailed 5-Fold OOF Classification Report (Best Model):
              precision    recall  f1-score   support

       anger     0.5641    0.6395    0.5994       674
 

In [18]:
import numpy as np
import pandas as pd
from scipy.sparse import hstack, csr_matrix
from sklearn.preprocessing import normalize, StandardScaler
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import f1_score, classification_report
from scipy.special import softmax

# 1. Row-wise L2 normalization on each block so no single feature block dominates
X_tr_w_norm = normalize(X_tr_word, norm="l2")
X_te_w_norm = normalize(X_te_word, norm="l2")
X_tr_c_norm = normalize(X_tr_char, norm="l2")
X_te_c_norm = normalize(X_te_char, norm="l2")

# Log-probabilities from DistilRoBERTa (7 classes: anger, disgust, fear, joy, neutral, sadness, surprise)
X_tr_logprob = np.log(np.clip(X_train_emo_prob, 1e-6, 1.0))
X_te_logprob = np.log(np.clip(X_test_emo_prob, 1e-6, 1.0))

scaler_lp = StandardScaler()
X_tr_lp_norm = normalize(scaler_lp.fit_transform(np.hstack([X_tr_logprob, X_train_emo_logit])))
X_te_lp_norm = normalize(scaler_lp.transform(np.hstack([X_te_logprob, X_test_emo_logit])))

# Block-weighted Hybrid Matrix
X_tr_opt = hstack([
    X_tr_w_norm * 1.0,
    X_tr_c_norm * 0.9,
    csr_matrix(X_train_minilm * 1.1),
    csr_matrix(X_train_emo_emb * 1.8),
    csr_matrix(X_tr_lp_norm * 1.4),
]).tocsr()

X_te_opt = hstack([
    X_te_w_norm * 1.0,
    X_te_c_norm * 0.9,
    csr_matrix(X_test_minilm * 1.1),
    csr_matrix(X_test_emo_emb * 1.8),
    csr_matrix(X_te_lp_norm * 1.4),
]).tocsr()

X_tr_dense_opt = np.hstack([X_train_emo_emb * 1.6, X_train_minilm * 1.0, X_tr_lp_norm * 1.3])
X_te_dense_opt = np.hstack([X_test_emo_emb * 1.6, X_test_minilm * 1.0, X_te_lp_norm * 1.3])

# Tempered class weights (alpha=0.55 balances precision & recall better than alpha=1.0 'balanced')
counts = np.bincount(y_enc)
total = len(y_enc)
tempered_weights = {c: (total / (n_classes * counts[c])) ** 0.55 for c in range(n_classes)}

oof_p1 = np.zeros((len(y_enc), n_classes))  # Calibrated LinearSVC (tempered weight)
oof_p2 = np.zeros((len(y_enc), n_classes))  # Calibrated LinearSVC (balanced weight)
oof_p3 = np.zeros((len(y_enc), n_classes))  # LogisticRegression (tempered weight, full hybrid)
oof_p4 = np.zeros((len(y_enc), n_classes))  # LogisticRegression (tempered weight, dense)
oof_p5 = np.zeros((len(y_enc), n_classes))  # Calibrated RidgeClassifier (tempered weight)

test_p1 = np.zeros((len(test_df), n_classes))
test_p2 = np.zeros((len(test_df), n_classes))
test_p3 = np.zeros((len(test_df), n_classes))
test_p4 = np.zeros((len(test_df), n_classes))
test_p5 = np.zeros((len(test_df), n_classes))

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_tr_opt, y_enc)):
    # M1: Calibrated LinearSVC (tempered class weights, C=0.12)
    m1 = CalibratedClassifierCV(LinearSVC(C=0.12, class_weight=tempered_weights, random_state=42 + fold), cv=3)
    m1.fit(X_tr_opt[tr_idx], y_enc[tr_idx])
    oof_p1[va_idx] = m1.predict_proba(X_tr_opt[va_idx])
    test_p1 += m1.predict_proba(X_te_opt) / skf.n_splits

    # M2: Calibrated LinearSVC (balanced class weights, C=0.10)
    m2 = CalibratedClassifierCV(LinearSVC(C=0.10, class_weight="balanced", random_state=42 + fold), cv=3)
    m2.fit(X_tr_opt[tr_idx], y_enc[tr_idx])
    oof_p2[va_idx] = m2.predict_proba(X_tr_opt[va_idx])
    test_p2 += m2.predict_proba(X_te_opt) / skf.n_splits

    # M3: LogisticRegression (tempered weights, C=0.4) on full hybrid
    m3 = LogisticRegression(C=0.4, class_weight=tempered_weights, max_iter=1000, random_state=42 + fold)
    m3.fit(X_tr_opt[tr_idx], y_enc[tr_idx])
    oof_p3[va_idx] = m3.predict_proba(X_tr_opt[va_idx])
    test_p3 += m3.predict_proba(X_te_opt) / skf.n_splits

    # M4: LogisticRegression (tempered weights, C=0.25) on dense features
    m4 = LogisticRegression(C=0.25, class_weight=tempered_weights, max_iter=1000, random_state=42 + fold)
    m4.fit(X_tr_dense_opt[tr_idx], y_enc[tr_idx])
    oof_p4[va_idx] = m4.predict_proba(X_tr_dense_opt[va_idx])
    test_p4 += m4.predict_proba(X_te_dense_opt) / skf.n_splits

    # M5: Calibrated RidgeClassifier (L2 linear model, alpha=3.0, tempered weights)
    m5 = CalibratedClassifierCV(RidgeClassifier(alpha=3.0, class_weight=tempered_weights, random_state=42 + fold), cv=3)
    m5.fit(X_tr_opt[tr_idx], y_enc[tr_idx])
    oof_p5[va_idx] = m5.predict_proba(X_tr_opt[va_idx])
    test_p5 += m5.predict_proba(X_te_opt) / skf.n_splits

for idx, (name, p) in enumerate([
    ("M1: Calibrated LinearSVC (Tempered)", oof_p1),
    ("M2: Calibrated LinearSVC (Balanced)", oof_p2),
    ("M3: LogisticRegression (Full Hybrid, Tempered)", oof_p3),
    ("M4: LogisticRegression (Dense Only, Tempered)", oof_p4),
    ("M5: Calibrated RidgeClassifier (Full Hybrid)", oof_p5),
], 1):
    print(f"{name:48s} -> 5-Fold Macro-F1: {f1_score(y_enc, p.argmax(axis=1), average='macro'):.4f}")

# Blend probabilities
oof_blend = 0.30 * oof_p1 + 0.20 * oof_p2 + 0.20 * oof_p3 + 0.15 * oof_p4 + 0.15 * oof_p5
test_blend = 0.30 * test_p1 + 0.20 * test_p2 + 0.20 * test_p3 + 0.15 * test_p4 + 0.15 * test_p5
f1_blend = f1_score(y_enc, oof_blend.argmax(axis=1), average="macro")

# Prior-matching threshold calibration (aligns predicted class prior with tempered target prior)
def tune_prior_power(oof_probs, y_true_enc):
    emp_prior = np.bincount(y_true_enc) / len(y_true_enc)
    best_gamma, best_f1 = 0.0, f1_score(y_true_enc, oof_probs.argmax(axis=1), average="macro")
    for gamma in np.linspace(-0.35, 0.35, 71):
        scale = emp_prior ** gamma
        score = f1_score(y_true_enc, (oof_probs * scale).argmax(axis=1), average="macro")
        if score > best_f1:
            best_f1 = score
            best_gamma = gamma
    return best_gamma, best_f1

best_gamma, f1_blend_prior = tune_prior_power(oof_blend, y_enc)
emp_prior = np.bincount(y_enc) / len(y_enc)
prior_scale = emp_prior ** best_gamma

print("-" * 72)
print(f"Refined 5-Model Blend Macro-F1           : {f1_blend:.4f}")
print(f"Refined Blend + Prior Scaling (g={best_gamma:+.2f}) : {f1_blend_prior:.4f}")
print("-" * 72)

best_final_oof = (oof_blend * prior_scale).argmax(axis=1)
print(classification_report(y_true, le.inverse_transform(best_final_oof), digits=4))

# Save both the clean Ensemble and the Refined Blend so we can compare/submit the highest CV one
if f1_blend_prior >= max(f1_ens_raw, f1_m_svc):
    best_test_preds = le.inverse_transform((test_blend * prior_scale).argmax(axis=1))
    best_cv_score = f1_blend_prior
    best_model_name = f"5-Model L2-Normalized Tempered Blend (CV={best_cv_score:.4f})"
else:
    best_test_preds = le.inverse_transform(test_ens_prob.argmax(axis=1))
    best_cv_score = f1_ens_raw
    best_model_name = f"4-Model Soft-Voting Ensemble (CV={best_cv_score:.4f})"

sub_improved = pd.DataFrame({
    "id": test_df["post_id"].astype(str),
    "emotion": best_test_preds
})
sub_improved.to_csv("submission_improved.csv", index=False)
print(f"Updated submission_improved.csv using: {best_model_name}")
print("Predicted test distribution:\n", sub_improved["emotion"].value_counts())

M1: Calibrated LinearSVC (Tempered)              -> 5-Fold Macro-F1: 0.6211
M2: Calibrated LinearSVC (Balanced)              -> 5-Fold Macro-F1: 0.6233
M3: LogisticRegression (Full Hybrid, Tempered)   -> 5-Fold Macro-F1: 0.6187
M4: LogisticRegression (Dense Only, Tempered)    -> 5-Fold Macro-F1: 0.6142
M5: Calibrated RidgeClassifier (Full Hybrid)     -> 5-Fold Macro-F1: 0.6198
------------------------------------------------------------------------
Refined 5-Model Blend Macro-F1           : 0.6220
Refined Blend + Prior Scaling (g=-0.02) : 0.6235
------------------------------------------------------------------------
              precision    recall  f1-score   support

       anger     0.5683    0.6484    0.6057       674
     disgust     0.5552    0.4427    0.4926       454
        fear     0.6741    0.6319    0.6523       288
         joy     0.7373    0.7634    0.7501       875
     sadness     0.6372    0.6433    0.6403       628
    surprise     0.6158    0.5850    0.6000       

In [19]:
sub_m2 = pd.DataFrame({
    "id": test_df["post_id"].astype(str),
    "emotion": le.inverse_transform(test_p2.argmax(axis=1))
})
sub_m2.to_csv("submission_m2_balanced.csv", index=False)
print("Saved backup candidate submission_m2_balanced.csv, distribution:\n", sub_m2["emotion"].value_counts())

Saved backup candidate submission_m2_balanced.csv, distribution:
 emotion
joy         390
anger       353
sadness     284
surprise    173
disgust     133
fear        107
Name: count, dtype: int64


In [20]:
import os
import time
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.preprocessing import normalize

torch.set_num_threads(min(16, os.cpu_count() or 8))
go_cache_path = "data/kaggle/cached_goemotions_features.npz"

train_texts_raw = train_df["text"].fillna("").astype(str).tolist()
test_texts_raw  = test_df["text"].fillna("").astype(str).tolist()

if os.path.exists(go_cache_path):
    print(f"Loading cached GoEmotions features from {go_cache_path}...")
    go_data = np.load(go_cache_path)
    X_train_go_emb   = go_data["X_train_go_emb"]
    X_test_go_emb    = go_data["X_test_go_emb"]
    X_train_go_prob  = go_data["X_train_go_prob"]
    X_test_go_prob   = go_data["X_test_go_prob"]
    X_train_go_logit = go_data["X_train_go_logit"]
    X_test_go_logit  = go_data["X_test_go_logit"]
else:
    t0 = time.time()
    go_model_name = "SamLowe/roberta-base-go_emotions"
    print(f"Extracting {go_model_name} (28 fine-grained Reddit emotions + 768-d emb)...")
    go_tokenizer = AutoTokenizer.from_pretrained(go_model_name)
    go_model = AutoModelForSequenceClassification.from_pretrained(go_model_name, output_hidden_states=True)
    go_model.eval()

    def extract_goemotions(texts, batch_size=64):
        all_emb, all_prob, all_logit = [], [], []
        for i in range(0, len(texts), batch_size):
            batch = texts[i : i + batch_size]
            enc = go_tokenizer(batch, padding=True, truncation=True, max_length=128, return_tensors="pt")
            with torch.inference_mode():
                out = go_model(**enc)
                logits = out.logits
                # GoEmotions is multi-label -> sigmoid captures independent fine-grained emotion intensities
                probs = torch.sigmoid(logits)
                last_hidden = out.hidden_states[-1]
                cls_emb = last_hidden[:, 0, :]
                mask = enc["attention_mask"].unsqueeze(-1).float()
                mean_emb = (last_hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)
                combined_emb = (cls_emb + mean_emb) * 0.5
            all_emb.append(combined_emb.numpy())
            all_prob.append(probs.numpy())
            all_logit.append(logits.numpy())
            if (i // batch_size + 1) % 15 == 0 or i + batch_size >= len(texts):
                print(f"  Processed {min(i + batch_size, len(texts))}/{len(texts)} posts ({time.time() - t0:.1f}s)")
        emb = normalize(np.vstack(all_emb)).astype(np.float32)
        prob = np.vstack(all_prob).astype(np.float32)
        logit = np.vstack(all_logit).astype(np.float32)
        return emb, prob, logit

    X_train_go_emb, X_train_go_prob, X_train_go_logit = extract_goemotions(train_texts_raw)
    X_test_go_emb,  X_test_go_prob,  X_test_go_logit  = extract_goemotions(test_texts_raw)

    np.savez_compressed(
        go_cache_path,
        X_train_go_emb=X_train_go_emb, X_test_go_emb=X_test_go_emb,
        X_train_go_prob=X_train_go_prob, X_test_go_prob=X_test_go_prob,
        X_train_go_logit=X_train_go_logit, X_test_go_logit=X_test_go_logit,
    )
    print(f"Saved cached GoEmotions features to {go_cache_path} in {time.time() - t0:.1f}s!")

print("GoEmotions shapes:", X_train_go_emb.shape, X_train_go_prob.shape)

Extracting SamLowe/roberta-base-go_emotions (28 fine-grained Reddit emotions + 768-d emb)...


config.json:   0%|          | 0.00/1.92k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/380 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.11M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  499MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

  Processed 960/3360 posts (218.5s)
  Processed 1920/3360 posts (301.7s)
  Processed 2880/3360 posts (384.9s)
  Processed 3360/3360 posts (420.2s)
  Processed 960/1440 posts (489.1s)
  Processed 1440/1440 posts (530.3s)
Saved cached GoEmotions features to data/kaggle/cached_goemotions_features.npz in 531.4s!
GoEmotions shapes: (3360, 768) (3360, 28)


In [21]:
import numpy as np
import pandas as pd
from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize, StandardScaler, LabelEncoder
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report

# Load both cached feature sets cleanly
c1 = np.load("data/kaggle/cached_emotion_features.npz")
X_train_minilm, X_test_minilm = c1["X_train_minilm"], c1["X_test_minilm"]
X_train_emo_emb, X_test_emo_emb = c1["X_train_emo_emb"], c1["X_test_emo_emb"]
X_train_emo_prob, X_test_emo_prob = c1["X_train_emo_prob"], c1["X_test_emo_prob"]
X_train_emo_logit, X_test_emo_logit = c1["X_train_emo_logit"], c1["X_test_emo_logit"]

c2 = np.load("data/kaggle/cached_goemotions_features.npz")
X_train_go_emb, X_test_go_emb = c2["X_train_go_emb"], c2["X_test_go_emb"]
X_train_go_prob, X_test_go_prob = c2["X_train_go_prob"], c2["X_test_go_prob"]
X_train_go_logit, X_test_go_logit = c2["X_train_go_logit"], c2["X_test_go_logit"]

train_texts_clean = train_df["clean_text"].fillna("").astype(str).tolist()
test_texts_clean  = test_df["clean_text"].fillna("").astype(str).tolist()
y_true = train_df["label"].astype(str).to_numpy()

le = LabelEncoder()
y_enc = le.fit_transform(y_true)
classes = le.classes_
n_classes = len(classes)

# 1. Row-wise L2-normalized Word + Char TF-IDF
word_vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=25000, sublinear_tf=True)
char_vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=3, max_features=35000, sublinear_tf=True)

X_tr_w = normalize(word_vec.fit_transform(train_texts_clean), norm="l2")
X_te_w = normalize(word_vec.transform(test_texts_clean), norm="l2")
X_tr_c = normalize(char_vec.fit_transform(train_texts_clean), norm="l2")
X_te_c = normalize(char_vec.transform(test_texts_clean), norm="l2")

# 2. Standardized + L2-normalized fine-grained emotion probabilities & logits
sc_hart = StandardScaler()
X_tr_hart_lp = normalize(sc_hart.fit_transform(np.hstack([np.log(np.clip(X_train_emo_prob, 1e-6, 1.0)), X_train_emo_logit])))
X_te_hart_lp = normalize(sc_hart.transform(np.hstack([np.log(np.clip(X_test_emo_prob, 1e-6, 1.0)), X_test_emo_logit])))

sc_go = StandardScaler()
X_tr_go_lp = normalize(sc_go.fit_transform(np.hstack([X_train_go_prob, X_train_go_logit * 0.25])))
X_te_go_lp = normalize(sc_go.transform(np.hstack([X_test_go_prob, X_test_go_logit * 0.25])))

# 3. 7-Block Dual-Transformer Hybrid Matrix
X_tr_dual = hstack([
    X_tr_w * 0.9,
    X_tr_c * 0.8,
    csr_matrix(X_train_minilm * 1.0),
    csr_matrix(X_train_emo_emb * 1.6),
    csr_matrix(X_tr_hart_lp * 1.3),
    csr_matrix(X_train_go_emb * 1.8),
    csr_matrix(X_tr_go_lp * 1.6),
]).tocsr()

X_te_dual = hstack([
    X_te_w * 0.9,
    X_te_c * 0.8,
    csr_matrix(X_test_minilm * 1.0),
    csr_matrix(X_test_emo_emb * 1.6),
    csr_matrix(X_te_hart_lp * 1.3),
    csr_matrix(X_test_go_emb * 1.8),
    csr_matrix(X_te_go_lp * 1.6),
]).tocsr()

# Dense-only Dual-Transformer Matrix
X_tr_dense_dual = np.hstack([
    X_train_minilm * 0.9,
    X_train_emo_emb * 1.5,
    X_tr_hart_lp * 1.2,
    X_train_go_emb * 1.8,
    X_tr_go_lp * 1.6,
])
X_te_dense_dual = np.hstack([
    X_test_minilm * 0.9,
    X_test_emo_emb * 1.5,
    X_te_hart_lp * 1.2,
    X_test_go_emb * 1.8,
    X_te_go_lp * 1.6,
])

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
counts = np.bincount(y_enc)
tempered_weights = {c: (len(y_enc) / (n_classes * counts[c])) ** 0.55 for c in range(n_classes)}

oof_d1 = np.zeros((len(y_enc), n_classes))  # Calibrated LinearSVC (Balanced, Dual Hybrid)
oof_d2 = np.zeros((len(y_enc), n_classes))  # Calibrated LinearSVC (Tempered, Dual Hybrid)
oof_d3 = np.zeros((len(y_enc), n_classes))  # LogisticRegression (Tempered, Dual Hybrid)
oof_d4 = np.zeros((len(y_enc), n_classes))  # LogisticRegression (Tempered, Dense Dual)
oof_d5 = np.zeros((len(y_enc), n_classes))  # Calibrated RidgeClassifier (Tempered, Dual Hybrid)
oof_d6 = np.zeros((len(y_enc), n_classes))  # Regularized MLP (Dense Dual)

test_d1 = np.zeros((len(test_df), n_classes))
test_d2 = np.zeros((len(test_df), n_classes))
test_d3 = np.zeros((len(test_df), n_classes))
test_d4 = np.zeros((len(test_df), n_classes))
test_d5 = np.zeros((len(test_df), n_classes))
test_d6 = np.zeros((len(test_df), n_classes))

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_tr_dual, y_enc)):
    m1 = CalibratedClassifierCV(LinearSVC(C=0.08, class_weight="balanced", random_state=42 + fold), cv=3)
    m1.fit(X_tr_dual[tr_idx], y_enc[tr_idx])
    oof_d1[va_idx] = m1.predict_proba(X_tr_dual[va_idx])
    test_d1 += m1.predict_proba(X_te_dual) / skf.n_splits

    m2 = CalibratedClassifierCV(LinearSVC(C=0.10, class_weight=tempered_weights, random_state=42 + fold), cv=3)
    m2.fit(X_tr_dual[tr_idx], y_enc[tr_idx])
    oof_d2[va_idx] = m2.predict_proba(X_tr_dual[va_idx])
    test_d2 += m2.predict_proba(X_te_dual) / skf.n_splits

    m3 = LogisticRegression(C=0.30, class_weight=tempered_weights, max_iter=1000, random_state=42 + fold)
    m3.fit(X_tr_dual[tr_idx], y_enc[tr_idx])
    oof_d3[va_idx] = m3.predict_proba(X_tr_dual[va_idx])
    test_d3 += m3.predict_proba(X_te_dual) / skf.n_splits

    m4 = LogisticRegression(C=0.20, class_weight=tempered_weights, max_iter=1000, random_state=42 + fold)
    m4.fit(X_tr_dense_dual[tr_idx], y_enc[tr_idx])
    oof_d4[va_idx] = m4.predict_proba(X_tr_dense_dual[va_idx])
    test_d4 += m4.predict_proba(X_te_dense_dual) / skf.n_splits

    m5 = CalibratedClassifierCV(RidgeClassifier(alpha=4.0, class_weight=tempered_weights, random_state=42 + fold), cv=3)
    m5.fit(X_tr_dual[tr_idx], y_enc[tr_idx])
    oof_d5[va_idx] = m5.predict_proba(X_tr_dual[va_idx])
    test_d5 += m5.predict_proba(X_te_dual) / skf.n_splits

    m6 = MLPClassifier(hidden_layer_sizes=(160,), alpha=2.0, max_iter=45, early_stopping=True, random_state=42 + fold)
    m6.fit(X_tr_dense_dual[tr_idx], y_enc[tr_idx])
    oof_d6[va_idx] = m6.predict_proba(X_tr_dense_dual[va_idx])
    test_d6 += m6.predict_proba(X_te_dense_dual) / skf.n_splits

print("=" * 74)
print("DUAL-TRANSFORMER (GoEmotions + DistilRoBERTa-Emotion) 5-FOLD CV SUMMARY")
print("=" * 74)
for name, p in [
    ("D1: Calibrated LinearSVC (Balanced, Dual Hybrid)", oof_d1),
    ("D2: Calibrated LinearSVC (Tempered, Dual Hybrid)", oof_d2),
    ("D3: LogisticRegression (Tempered, Dual Hybrid)", oof_d3),
    ("D4: LogisticRegression (Tempered, Dense Dual)", oof_d4),
    ("D5: Calibrated RidgeClassifier (Dual Hybrid)", oof_d5),
    ("D6: Regularized MLP (Dense Dual)", oof_d6),
]:
    print(f"{name:50s} -> Macro-F1: {f1_score(y_enc, p.argmax(axis=1), average='macro'):.4f}")

# Soft-voting ensemble across all 6 diverse models
oof_dual_blend = (
    0.25 * oof_d1
    + 0.25 * oof_d2
    + 0.18 * oof_d3
    + 0.12 * oof_d4
    + 0.12 * oof_d5
    + 0.08 * oof_d6
)
test_dual_blend = (
    0.25 * test_d1
    + 0.25 * test_d2
    + 0.18 * test_d3
    + 0.12 * test_d4
    + 0.12 * test_d5
    + 0.08 * test_d6
)

f1_dual_raw = f1_score(y_enc, oof_dual_blend.argmax(axis=1), average="macro")

# Nested OOF prior-power tuning (evaluated out-of-fold so CV is completely leak-free)
emp_prior = counts / len(y_enc)
oof_nested_preds = np.zeros(len(y_enc), dtype=int)
gammas = []
for tr_idx, va_idx in skf.split(oof_dual_blend, y_enc):
    best_g, best_s = 0.0, -1.0
    for g in np.linspace(-0.30, 0.30, 61):
        s = f1_score(y_enc[tr_idx], (oof_dual_blend[tr_idx] * (emp_prior ** g)).argmax(axis=1), average="macro")
        if s > best_s:
            best_s, best_g = s, g
    gammas.append(best_g)
    oof_nested_preds[va_idx] = (oof_dual_blend[va_idx] * (emp_prior ** best_g)).argmax(axis=1)

f1_dual_nested = f1_score(y_enc, oof_nested_preds, average="macro")
mean_gamma = float(np.mean(gammas))

print("-" * 74)
print(f"6-Model Dual-Transformer Blend (Raw Argmax)      : {f1_dual_raw:.4f} (vs prev 0.6235: {f1_dual_raw - 0.6235:+.4f})")
print(f"6-Model Blend + Nested OOF Prior Scaling (g={mean_gamma:+.3f}): {f1_dual_nested:.4f} (vs prev 0.6235: {f1_dual_nested - 0.6235:+.4f})")
print("-" * 74)

if f1_dual_nested >= f1_dual_raw:
    final_oof = oof_nested_preds
    final_test_preds = le.inverse_transform((test_dual_blend * (emp_prior ** mean_gamma)).argmax(axis=1))
    chosen_desc = f"Dual-Transformer 6-Model Blend + Prior Scaling (CV={f1_dual_nested:.4f})"
else:
    final_oof = oof_dual_blend.argmax(axis=1)
    final_test_preds = le.inverse_transform(test_dual_blend.argmax(axis=1))
    chosen_desc = f"Dual-Transformer 6-Model Blend (CV={f1_dual_raw:.4f})"

print(classification_report(y_true, le.inverse_transform(final_oof), digits=4))

sub_go = pd.DataFrame({
    "id": test_df["post_id"].astype(str),
    "emotion": final_test_preds
})
sub_go.to_csv("submission_goemotions_blend.csv", index=False)
print(f"Saved submission_goemotions_blend.csv ({chosen_desc}), shape={sub_go.shape}")
print("Predicted test distribution:\n", sub_go["emotion"].value_counts())

DUAL-TRANSFORMER (GoEmotions + DistilRoBERTa-Emotion) 5-FOLD CV SUMMARY
D1: Calibrated LinearSVC (Balanced, Dual Hybrid)   -> Macro-F1: 0.6400
D2: Calibrated LinearSVC (Tempered, Dual Hybrid)   -> Macro-F1: 0.6391
D3: LogisticRegression (Tempered, Dual Hybrid)     -> Macro-F1: 0.6351
D4: LogisticRegression (Tempered, Dense Dual)      -> Macro-F1: 0.6313
D5: Calibrated RidgeClassifier (Dual Hybrid)       -> Macro-F1: 0.6414
D6: Regularized MLP (Dense Dual)                   -> Macro-F1: 0.6339
--------------------------------------------------------------------------
6-Model Dual-Transformer Blend (Raw Argmax)      : 0.6415 (vs prev 0.6235: +0.0180)
6-Model Blend + Nested OOF Prior Scaling (g=+0.238): 0.6441 (vs prev 0.6235: +0.0206)
--------------------------------------------------------------------------
              precision    recall  f1-score   support

       anger     0.5775    0.6691    0.6199       674
     disgust     0.5793    0.4185    0.4859       454
        fear     0.

## Final 100% Compliant 9-Family Clean Pipeline

The complete GPU training and ensembling code for our final submissions is archived in **`DM2026-Lab2-Kaggle-Clean.ipynb`** (executed on Google Colab Pro with an L4 GPU). All nine model families rely exclusively on competition training labels and OSI-approved (MIT / Apache 2.0) general-purpose pre-trained models, with zero external emotion supervision.

### Iterative `.npz` Probability Handoff (Day 1 $\rightarrow$ Day 4)
To stay within daily GPU and submission limits while keeping all out-of-fold (OOF) and test probability predictions strictly aligned under the same 5-fold `StratifiedKFold(n_splits=5, shuffle=True, random_state=42)` split, we saved each stage's calibrated probabilities into `.npz` checkpoints and incrementally expanded the ensemble:

1. **Day 1 (4-Family Clean Base):** Trained `RoBERTa-base` (`0.5946`), `DeBERTa-v3-base` (`0.5432`, pure `FP32` to prevent `FP16` overflow), `BGE-Large-v1.5` (`0.5623`), and lexical Word+Char TF-IDF (`0.5087`), achieving `0.6119` OOF (`0.5912` Public LB).
2. **Day 2 (6-Family Expansion):** Added `RoBERTa-large` Seed 1 (`0.6015`) and `DeBERTa-v3-large` (`0.4749`) via LLRD + Mean/Max Pooling + 5x Multi-Sample Dropout, lifting the arithmetic blend to `0.6165` OOF (`0.6127` Public LB).
3. **Day 3 (7-Family Expansion, `clean_day3_all_probs.npz`):** Added `intfloat/e5-large-v2` embeddings (`0.5473`), pushing the balanced arithmetic blend to `0.6175` OOF and `0.6189` Public LB.
4. **Day 4 (Final 9-Family Grand Ensemble, `clean_day4_all_probs.npz`):** Loaded Day 3 probabilities and added `RoBERTa-large` Seed 2 (`seed=777`, `0.6100` OOF) and `sentence-transformers/all-mpnet-base-v2` (`0.4980` OOF).
   * **Final Selection #1 (`sub_day4_2_9fam_arith_raw.csv`):** 9-Family Balanced Arithmetic Soft-Voting (Raw Argmax) — **`0.6193` OOF / `0.6202` Public LB**.
   * **Final Selection #2 (`sub_day4_5_9fam_meta_l2.csv`):** 9-Family L2 Meta-Stacker (`LogisticRegression`, `C=0.10`, strict 5-fold nested OOF) — **`0.6201` OOF / `0.6150` Public LB**.

### Day 1: Environment Setup (Uploading Data to Colab, GPU)

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving clean_day1_all_probs.npz to clean_day1_all_probs.npz
Saving data_identification.csv to data_identification.csv
Saving emotion.csv to emotion.csv
Saving final_posts.json to final_posts.json
Saving sampleSubmission.csv to sampleSubmission.csv


In [ ]:
!nvidia-smi
!pip install -q -U transformers accelerate scikit-learn

Wed Nov 18 23:43:46 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P0             46W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

### Day 1: Preprocessing (light_clean), TF-IDF, RoBERTa-base, DeBERTa-v3-base. Note: In the output for this entry, training DeBERTa using fp16 failed (accuracy 0.20).

In [ ]:
import os
import re
import json
import time
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, AutoConfig, get_cosine_schedule_with_warmup
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder, normalize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import f1_score, classification_report
from scipy.sparse import hstack

# ==========================================
# 1. Reproducibility & Device
# ==========================================
SEED = 42
def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ==========================================
# 2. Load Strictly In-Competition Data
# ==========================================
with open("final_posts.json", "r", encoding="utf-8") as f:
    raw_posts = json.load(f)

if isinstance(raw_posts, dict):
    if all(isinstance(v, str) for v in list(raw_posts.values())[:5]):
        records = [{"post_id": str(k), "text": v} for k, v in raw_posts.items()]
    else:
        first_val = next(iter(raw_posts.values()))
        raw_posts = first_val if isinstance(first_val, list) else list(raw_posts.values())

if isinstance(raw_posts, list):
    records = []
    for item in raw_posts:
        post = item["root"]["_source"]["post"] if "root" in item else item
        pid = post.get("post_id") or post.get("id")
        txt = post.get("text") or post.get("content") or ""
        records.append({"post_id": str(pid), "text": txt})

posts_df = pd.DataFrame(records)
ident_df = pd.read_csv("data_identification.csv", dtype=str)
emo_df   = pd.read_csv("emotion.csv", dtype=str)
full_df  = posts_df.merge(ident_df, on="post_id", how="inner")

def light_clean(text):
    text = str(text)
    text = re.sub(r"http\S+|www\.\S+", "http", text)
    text = re.sub(r"@\w+", "@user", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

full_df["clean_text"] = full_df["text"].apply(light_clean)
train_df = full_df[full_df["set"] == "train"].merge(emo_df, on="post_id", how="inner").reset_index(drop=True)
test_df  = full_df[full_df["set"] == "test"].reset_index(drop=True)

sample_sub = pd.read_csv("sampleSubmission.csv", dtype=str)
sub_id_col = sample_sub.columns[0]
test_df = sample_sub[[sub_id_col]].merge(test_df, left_on=sub_id_col, right_on="post_id", how="left")

le = LabelEncoder()
y_all = le.fit_transform(train_df["label"].astype(str))
classes = le.classes_
num_classes = len(classes)
counts = np.bincount(y_all)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

# ==========================================
# 3. Model Family A: Clean Lexical Word+Char TF-IDF Calibrated Ensemble
# ==========================================
print("\n[1/3] Training Clean Lexical TF-IDF Calibrated Ensemble...")
word_vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=30000, sublinear_tf=True)
char_vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=3, max_features=45000, sublinear_tf=True)

X_tr_lex = hstack([
    normalize(word_vec.fit_transform(train_df["clean_text"])),
    normalize(char_vec.fit_transform(train_df["clean_text"])),
]).tocsr()
X_te_lex = hstack([
    normalize(word_vec.transform(test_df["clean_text"])),
    normalize(char_vec.transform(test_df["clean_text"])),
]).tocsr()

tempered_w = {c: (len(y_all) / (num_classes * counts[c])) ** 0.5 for c in range(num_classes)}
oof_lex = np.zeros((len(train_df), num_classes), dtype=np.float32)
test_lex = np.zeros((len(test_df), num_classes), dtype=np.float32)

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_tr_lex, y_all)):
    m_svc = CalibratedClassifierCV(LinearSVC(C=0.25, class_weight=tempered_w, random_state=SEED + fold), cv=3)
    m_lr  = LogisticRegression(C=1.5, class_weight=tempered_w, max_iter=1000, random_state=SEED + fold)
    m_svc.fit(X_tr_lex[tr_idx], y_all[tr_idx])
    m_lr.fit(X_tr_lex[tr_idx], y_all[tr_idx])
    oof_lex[va_idx] = 0.5 * m_svc.predict_proba(X_tr_lex[va_idx]) + 0.5 * m_lr.predict_proba(X_tr_lex[va_idx])
    test_lex += (0.5 * m_svc.predict_proba(X_te_lex) + 0.5 * m_lr.predict_proba(X_te_lex)) / 5.0

print(f"Clean Lexical TF-IDF 5-Fold CV Macro-F1: {f1_score(y_all, oof_lex.argmax(axis=1), average='macro'):.4f}")

# ==========================================
# 4. Custom Mean+Max Pooling + Multi-Sample Dropout Transformer Classifier
# ==========================================
class MeanMaxPoolClassifier(nn.Module):
    def __init__(self, model_name, num_classes):
        super().__init__()
        self.config = AutoConfig.from_pretrained(model_name)
        self.config.update({"hidden_dropout_prob": 0.1, "attention_probs_dropout_prob": 0.1})
        self.backbone = AutoModel.from_pretrained(model_name, config=self.config)
        hidden = self.config.hidden_size
        self.dropouts = nn.ModuleList([nn.Dropout(0.1 * i) for i in range(1, 6)])
        self.fc = nn.Linear(hidden * 2, num_classes)

    def forward(self, input_ids, attention_mask):
        out = self.backbone(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        mask = attention_mask.unsqueeze(-1).float()
        mean_pool = (out * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)
        max_pool = out.masked_fill(mask == 0, -1e4).max(dim=1).values
        feat = torch.cat([mean_pool, max_pool], dim=-1)
        logits = sum(self.fc(drop(feat)) for drop in self.dropouts) / len(self.dropouts)
        return logits

def train_clean_backbone(model_name, lr=2e-5, epochs=4, batch_size=32, max_len=144):
    print(f"\nTraining 5-Fold Clean Backbone: {model_name} (Pure MLM/RTD, zero emotion supervision)...")
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    class TextDS(Dataset):
        def __init__(self, texts, labels=None):
            self.enc = tokenizer(list(texts), truncation=True, max_length=max_len, padding=False)
            self.labels = list(labels) if labels is not None else None
        def __len__(self):
            return len(self.enc["input_ids"])
        def __getitem__(self, i):
            d = {"input_ids": self.enc["input_ids"][i], "attention_mask": self.enc["attention_mask"][i]}
            if self.labels is not None:
                d["labels"] = self.labels[i]
            return d

    def collate(batch):
        ml = max(len(x["input_ids"]) for x in batch)
        pad_id = tokenizer.pad_token_id or 0
        ids, masks, labs = [], [], []
        for x in batch:
            pl = ml - len(x["input_ids"])
            ids.append(x["input_ids"] + [pad_id] * pl)
            masks.append(x["attention_mask"] + [0] * pl)
            if "labels" in x:
                labs.append(x["labels"])
        out = {"input_ids": torch.tensor(ids, dtype=torch.long), "attention_mask": torch.tensor(masks, dtype=torch.long)}
        if labs:
            out["labels"] = torch.tensor(labs, dtype=torch.long)
        return out

    test_loader = DataLoader(TextDS(test_df["clean_text"]), batch_size=64, shuffle=False, collate_fn=collate)
    class_weights = torch.tensor([(len(y_all) / (num_classes * counts[c])) ** 0.45 for c in range(num_classes)], dtype=torch.float, device=device)

    oof_p = np.zeros((len(train_df), num_classes), dtype=np.float32)
    test_p = np.zeros((len(test_df), num_classes), dtype=np.float32)

    for fold, (tr_idx, va_idx) in enumerate(skf.split(train_df, y_all), 1):
        t0 = time.time()
        seed_everything(SEED + fold * 10)
        tr_loader = DataLoader(TextDS(train_df.loc[tr_idx, "clean_text"], y_all[tr_idx]), batch_size=batch_size, shuffle=True, collate_fn=collate)
        va_loader = DataLoader(TextDS(train_df.loc[va_idx, "clean_text"], y_all[va_idx]), batch_size=64, shuffle=False, collate_fn=collate)

        model = MeanMaxPoolClassifier(model_name, num_classes).to(device)
        optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
        total_steps = len(tr_loader) * epochs
        scheduler = get_cosine_schedule_with_warmup(optimizer, int(0.1 * total_steps), total_steps)
        criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.05)
        # DeBERTa-v3 prefers bf16 on A100/L4 or fp32/fp16 carefully
        use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
        amp_dtype = torch.bfloat16 if use_bf16 else torch.float16
        scaler = torch.amp.GradScaler("cuda", enabled=(not use_bf16 and device.type == "cuda"))

        best_f1, best_va, best_state = -1.0, None, None
        for ep in range(1, epochs + 1):
            model.train()
            for b in tr_loader:
                optimizer.zero_grad()
                with torch.amp.autocast("cuda", dtype=amp_dtype, enabled=(device.type == "cuda")):
                    logits = model(b["input_ids"].to(device), b["attention_mask"].to(device))
                    loss = criterion(logits, b["labels"].to(device))
                if scaler.is_enabled():
                    scaler.scale(loss).backward()
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    optimizer.step()
                scheduler.step()

            model.eval()
            va_list = []
            with torch.inference_mode():
                for b in va_loader:
                    with torch.amp.autocast("cuda", dtype=amp_dtype, enabled=(device.type == "cuda")):
                        logits = model(b["input_ids"].to(device), b["attention_mask"].to(device))
                        va_list.append(torch.softmax(logits.float(), dim=-1).cpu().numpy())
            va_probs = np.vstack(va_list)
            f1_ep = f1_score(y_all[va_idx], va_probs.argmax(axis=1), average="macro")
            if f1_ep > best_f1:
                best_f1, best_va = f1_ep, va_probs
                best_state = {k: v.cpu() for k, v in model.state_dict().items()}

        oof_p[va_idx] = best_va
        model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
        model.eval()
        te_list = []
        with torch.inference_mode():
            for b in test_loader:
                with torch.amp.autocast("cuda", dtype=amp_dtype, enabled=(device.type == "cuda")):
                    logits = model(b["input_ids"].to(device), b["attention_mask"].to(device))
                    te_list.append(torch.softmax(logits.float(), dim=-1).cpu().numpy())
        test_p += np.vstack(te_list) / 5.0
        print(f"  Fold {fold}/5 | Val Macro-F1: {best_f1:.4f} | {time.time() - t0:.1f}s")

    cv_f1 = f1_score(y_all, oof_p.argmax(axis=1), average="macro")
    print(f"--> {model_name} 5-Fold OOF Macro-F1: {cv_f1:.4f}")
    return oof_p, test_p

# Train two pure pre-trained backbones (zero external emotion labels)
oof_deb, test_deb = train_clean_backbone("microsoft/deberta-v3-base", lr=2.2e-5, epochs=4, batch_size=32)
oof_rob, test_rob = train_clean_backbone("FacebookAI/roberta-base", lr=2.5e-5, epochs=4, batch_size=32)

# ==========================================
# 5. Heterogeneous Ensemble + Nested OOF Class-Multiplier Tuning
# ==========================================
oof_ens = 0.50 * oof_deb + 0.35 * oof_rob + 0.15 * oof_lex
test_ens = 0.50 * test_deb + 0.35 * test_rob + 0.15 * test_lex
f1_ens_raw = f1_score(y_all, oof_ens.argmax(axis=1), average="macro")

# Nested OOF coordinate ascent for class multipliers (strictly leak-free)
def fit_multipliers(probs, labels, steps=2):
    mult = np.ones(num_classes, dtype=np.float32)
    best = f1_score(labels, (probs * mult).argmax(axis=1), average="macro")
    for _ in range(steps):
        for c in range(num_classes):
            best_m = mult[c]
            for cand in np.linspace(0.75, 1.35, 31):
                trial = mult.copy()
                trial[c] = cand
                s = f1_score(labels, (probs * trial).argmax(axis=1), average="macro")
                if s > best + 1e-5:
                    best, best_m = s, cand
            mult[c] = best_m
    return mult

oof_ens_opt = np.zeros(len(y_all), dtype=int)
fold_mults = []
for tr_idx, va_idx in skf.split(oof_ens, y_all):
    m = fit_multipliers(oof_ens[tr_idx], y_all[tr_idx])
    fold_mults.append(m)
    oof_ens_opt[va_idx] = (oof_ens[va_idx] * m).argmax(axis=1)

f1_ens_opt = f1_score(y_all, oof_ens_opt, average="macro")
mean_mult = np.mean(fold_mults, axis=0)

print("\n" + "=" * 74)
print("STRICTLY COMPLIANT HETEROGENEOUS ENSEMBLE SUMMARY")
print("=" * 74)
print(f"1. Clean Lexical TF-IDF Calibrated Ensemble  : {f1_score(y_all, oof_lex.argmax(axis=1), average='macro'):.4f}")
print(f"2. Clean 5-Fold RoBERTa-base (Mean+Max Pool) : {f1_score(y_all, oof_rob.argmax(axis=1), average='macro'):.4f}")
print(f"3. Clean 5-Fold DeBERTa-v3-base (Mean+Max)   : {f1_score(y_all, oof_deb.argmax(axis=1), average='macro'):.4f}")
print(f"4. 3-Family Heterogeneous Blend (Raw Argmax) : {f1_ens_raw:.4f}")
print(f"5. 3-Family Blend + Nested OOF Multipliers   : {f1_ens_opt:.4f}")
print("Learned Mean Class Multipliers:", dict(zip(classes, np.round(mean_mult, 3))))
print("=" * 74)

best_oof = oof_ens_opt if f1_ens_opt > f1_ens_raw else oof_ens.argmax(axis=1)
print(classification_report(train_df["label"], le.inverse_transform(best_oof), digits=4))

# Save both clean submissions in Kaggle's exact format (id, emotion)
sub_clean_v2 = pd.DataFrame({
    "id": test_df[sub_id_col],
    "emotion": le.inverse_transform(test_ens.argmax(axis=1))
})
sub_clean_v2.to_csv("submission_clean_heterogeneous_blend.csv", index=False)

sub_clean_v2_opt = pd.DataFrame({
    "id": test_df[sub_id_col],
    "emotion": le.inverse_transform((test_ens * mean_mult).argmax(axis=1))
})
sub_clean_v2_opt.to_csv("submission_clean_heterogeneous_opt.csv", index=False)

from google.colab import files
files.download("submission_clean_heterogeneous_blend.csv")
files.download("submission_clean_heterogeneous_opt.csv")

Using device: cuda

[1/3] Training Clean Lexical TF-IDF Calibrated Ensemble...
Clean Lexical TF-IDF 5-Fold CV Macro-F1: 0.5087

Training 5-Fold Clean Backbone: microsoft/deberta-v3-base (Pure MLM/RTD, zero emotion supervision)...


config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 2.46MB            

spm.model: downloading bytes:           |  0.00B            

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  371MB            

... (34 outputs left out)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd
from google.colab import files

df_clean = pd.read_csv("submission_clean_roberta_blend.csv")
df_clean.columns = ["id", "emotion"]
df_clean.to_csv("submission_clean_roberta_blend.csv", index=False)

print(df_clean.head(3))
files.download("submission_clean_roberta_blend.csv")

        id   emotion
0  xyzx9jp  surprise
1  l3vzzxt   sadness
2  g680iug   sadness


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Day 1: DeBERTa-v3-base converted to FP32; ensemble of 3 models.

In [ ]:
# ==========================================
# Fix DeBERTa-v3 (Explicit torch_dtype=torch.float32) & Blend with Saved RoBERTa + TF-IDF
# ==========================================
from transformers import AutoModelForSequenceClassification

def train_clean_deberta_fp32(model_name="microsoft/deberta-v3-base", lr=2e-5, epochs=4, batch_size=32, max_len=128):
    print(f"\nTraining 5-Fold {model_name} in explicit FP32...")
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    class TextDS(Dataset):
        def __init__(self, texts, labels=None):
            self.enc = tokenizer(list(texts), truncation=True, max_length=max_len, padding=False)
            self.labels = list(labels) if labels is not None else None
        def __len__(self):
            return len(self.enc["input_ids"])
        def __getitem__(self, i):
            d = {"input_ids": self.enc["input_ids"][i], "attention_mask": self.enc["attention_mask"][i]}
            if self.labels is not None:
                d["labels"] = self.labels[i]
            return d

    def collate(batch):
        ml = max(len(x["input_ids"]) for x in batch)
        pad_id = tokenizer.pad_token_id or 0
        ids, masks, labs = [], [], []
        for x in batch:
            pl = ml - len(x["input_ids"])
            ids.append(x["input_ids"] + [pad_id] * pl)
            masks.append(x["attention_mask"] + [0] * pl)
            if "labels" in x:
                labs.append(x["labels"])
        out = {"input_ids": torch.tensor(ids, dtype=torch.long), "attention_mask": torch.tensor(masks, dtype=torch.long)}
        if labs:
            out["labels"] = torch.tensor(labs, dtype=torch.long)
        return out

    test_loader = DataLoader(TextDS(test_df["clean_text"]), batch_size=64, shuffle=False, collate_fn=collate)
    class_weights = torch.tensor([(len(y_all) / (num_classes * counts[c])) ** 0.45 for c in range(num_classes)], dtype=torch.float32, device=device)

    oof_p = np.zeros((len(train_df), num_classes), dtype=np.float32)
    test_p = np.zeros((len(test_df), num_classes), dtype=np.float32)

    for fold, (tr_idx, va_idx) in enumerate(skf.split(train_df, y_all), 1):
        t0 = time.time()
        seed_everything(SEED + fold * 10)
        tr_loader = DataLoader(TextDS(train_df.loc[tr_idx, "clean_text"], y_all[tr_idx]), batch_size=batch_size, shuffle=True, collate_fn=collate)
        va_loader = DataLoader(TextDS(train_df.loc[va_idx, "clean_text"], y_all[va_idx]), batch_size=64, shuffle=False, collate_fn=collate)

        model = AutoModelForSequenceClassification.from_pretrained(
            model_name,
            num_labels=num_classes,
            hidden_dropout_prob=0.1,
            attention_probs_dropout_prob=0.1,
            torch_dtype=torch.float32
        ).float().to(device)

        optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
        total_steps = len(tr_loader) * epochs
        scheduler = get_cosine_schedule_with_warmup(optimizer, int(0.1 * total_steps), total_steps)
        criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.05)

        best_f1, best_va, best_state = -1.0, None, None
        for ep in range(1, epochs + 1):
            model.train()
            for b in tr_loader:
                optimizer.zero_grad()
                logits = model(input_ids=b["input_ids"].to(device), attention_mask=b["attention_mask"].to(device)).logits.float()
                loss = criterion(logits, b["labels"].to(device))
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                scheduler.step()

            model.eval()
            va_list = []
            with torch.inference_mode():
                for b in va_loader:
                    logits = model(input_ids=b["input_ids"].to(device), attention_mask=b["attention_mask"].to(device)).logits.float()
                    va_list.append(torch.softmax(logits, dim=-1).cpu().numpy())
            va_probs = np.vstack(va_list)
            f1_ep = f1_score(y_all[va_idx], va_probs.argmax(axis=1), average="macro")
            if f1_ep > best_f1:
                best_f1, best_va = f1_ep, va_probs
                best_state = {k: v.cpu() for k, v in model.state_dict().items()}

        oof_p[va_idx] = best_va
        model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
        model.eval()
        te_list = []
        with torch.inference_mode():
            for b in test_loader:
                logits = model(input_ids=b["input_ids"].to(device), attention_mask=b["attention_mask"].to(device)).logits.float()
                te_list.append(torch.softmax(logits, dim=-1).cpu().numpy())
        test_p += np.vstack(te_list) / 5.0
        print(f"  Fold {fold}/5 | Val Macro-F1: {best_f1:.4f} | {time.time() - t0:.1f}s")

    print(f"--> {model_name} (FP32) 5-Fold OOF Macro-F1: {f1_score(y_all, oof_p.argmax(axis=1), average='macro'):.4f}")
    return oof_p, test_p

oof_deb_ok, test_deb_ok = train_clean_deberta_fp32("microsoft/deberta-v3-base", lr=2e-5, epochs=4, batch_size=32)

# Grid-search optimal OOF blend weights across DeBERTa-v3 + RoBERTa-base + Lexical TF-IDF
best_w, best_blend_f1 = (0.5, 0.4, 0.1), -1.0
for wd in np.linspace(0.35, 0.65, 7):
    for wr in np.linspace(0.25, 0.55, 7):
        wl = round(1.0 - wd - wr, 4)
        if wl < 0:
            continue
        p = wd * oof_deb_ok + wr * oof_rob + wl * oof_lex
        s = f1_score(y_all, p.argmax(axis=1), average="macro")
        if s > best_blend_f1:
            best_blend_f1 = s
            best_w = (round(wd, 3), round(wr, 3), wl)

wd, wr, wl = best_w
oof_clean_tri = wd * oof_deb_ok + wr * oof_rob + wl * oof_lex
test_clean_tri = wd * test_deb_ok + wr * test_rob + wl * test_lex

# Nested OOF class-multiplier tuning
oof_tri_opt = np.zeros(len(y_all), dtype=int)
fold_mults = []
for tr_idx, va_idx in skf.split(oof_clean_tri, y_all):
    m = fit_multipliers(oof_clean_tri[tr_idx], y_all[tr_idx])
    fold_mults.append(m)
    oof_tri_opt[va_idx] = (oof_clean_tri[va_idx] * m).argmax(axis=1)

f1_tri_opt = f1_score(y_all, oof_tri_opt, average="macro")
mean_mult = np.mean(fold_mults, axis=0)

print("\n" + "=" * 74)
print("FIXED 100% COMPLIANT TRI-FAMILY ENSEMBLE SUMMARY")
print("=" * 74)
print(f"1. Clean Lexical TF-IDF Calibrated Ensemble  : {f1_score(y_all, oof_lex.argmax(axis=1), average='macro'):.4f}")
print(f"2. Clean 5-Fold RoBERTa-base (Mean+Max Pool) : {f1_score(y_all, oof_rob.argmax(axis=1), average='macro'):.4f}")
print(f"3. Clean 5-Fold DeBERTa-v3-base (FP32)       : {f1_score(y_all, oof_deb_ok.argmax(axis=1), average='macro'):.4f}")
print(f"4. Tri-Family Blend {best_w} (Raw)  : {best_blend_f1:.4f}")
print(f"5. Tri-Family Blend + Nested OOF Multipliers   : {f1_tri_opt:.4f}")
print("=" * 74)

sub_tri_raw = pd.DataFrame({"id": test_df[sub_id_col], "emotion": le.inverse_transform(test_clean_tri.argmax(axis=1))})
sub_tri_raw.to_csv("submission_clean_trifamily_blend.csv", index=False)

sub_tri_opt = pd.DataFrame({"id": test_df[sub_id_col], "emotion": le.inverse_transform((test_clean_tri * mean_mult).argmax(axis=1))})
sub_tri_opt.to_csv("submission_clean_trifamily_opt.csv", index=False)

from google.colab import files
files.download("submission_clean_trifamily_blend.csv")
files.download("submission_clean_trifamily_opt.csv")


Training 5-Fold microsoft/deberta-v3-base in explicit FP32...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
pooler.dense.bias                       | MISSING    | 
classifier.bias                         | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.den

  Fold 1/5 | Val Macro-F1: 0.5339 | 55.3s


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

... (11 outputs left out)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Day 1: BGE sentence embeddings and 4-family mixture; saved as `clean_day1_all_probs.npz`.

In [ ]:
!pip install -q sentence-transformers

from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import normalize
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import f1_score
from google.colab import files
import numpy as np
import pandas as pd

# 1. Extract pure general-purpose semantic embeddings (BAAI/bge-large-en-v1.5, MIT license, zero emotion labels)
bge_model = SentenceTransformer("BAAI/bge-large-en-v1.5", device=str(device))
X_tr_bge = normalize(bge_model.encode(train_df["clean_text"].tolist(), batch_size=64, show_progress_bar=False))
X_te_bge = normalize(bge_model.encode(test_df["clean_text"].tolist(), batch_size=64, show_progress_bar=False))

tempered_w = {c: (len(y_all) / (num_classes * counts[c])) ** 0.45 for c in range(num_classes)}
oof_bge = np.zeros((len(train_df), num_classes), dtype=np.float32)
test_bge = np.zeros((len(test_df), num_classes), dtype=np.float32)

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_tr_bge, y_all)):
    m_svc = CalibratedClassifierCV(LinearSVC(C=0.35, class_weight=tempered_w, random_state=SEED + fold), cv=3)
    m_lr  = LogisticRegression(C=2.0, class_weight=tempered_w, max_iter=1000, random_state=SEED + fold)
    m_svc.fit(X_tr_bge[tr_idx], y_all[tr_idx])
    m_lr.fit(X_tr_bge[tr_idx], y_all[tr_idx])
    oof_bge[va_idx] = 0.5 * m_svc.predict_proba(X_tr_bge[va_idx]) + 0.5 * m_lr.predict_proba(X_tr_bge[va_idx])
    test_bge += (0.5 * m_svc.predict_proba(X_te_bge) + 0.5 * m_lr.predict_proba(X_te_bge)) / 5.0

print(f"Clean BGE-Large-v1.5 5-Fold CV Macro-F1: {f1_score(y_all, oof_bge.argmax(axis=1), average='macro'):.4f}")

# 2. Pure Raw Soft-Voting Blend (No threshold/multiplier overfitting!)
# Combines 4 distinct clean model families: RoBERTa-base + DeBERTa-v3-base + BGE-Large + Lexical TF-IDF
weights_list = [
    ("Balanced_4Family", (0.35, 0.25, 0.25, 0.15)),  # (RoBERTa, DeBERTa, BGE-Large, Lexical)
    ("DenseHeavy_4Family", (0.38, 0.27, 0.25, 0.10)),
]

for name, (wr, wd, wb, wl) in weights_list:
    oof_4f = wr * oof_rob + wd * oof_deb_ok + wb * oof_bge + wl * oof_lex
    f1_4f = f1_score(y_all, oof_4f.argmax(axis=1), average="macro")
    print(f"{name} {(wr, wd, wb, wl)} -> 5-Fold OOF Raw Macro-F1: {f1_4f:.4f}")

# Pick the best raw blend on OOF
best_4f_name, (wr, wd, wb, wl) = max(
    weights_list,
    key=lambda item: f1_score(
        y_all,
        (item[1][0]*oof_rob + item[1][1]*oof_deb_ok + item[1][2]*oof_bge + item[1][3]*oof_lex).argmax(axis=1),
        average="macro"
    )
)

test_4f = wr * test_rob + wd * test_deb_ok + wb * test_bge + wl * test_lex
sub_4f = pd.DataFrame({
    "id": test_df[sub_id_col],
    "emotion": le.inverse_transform(test_4f.argmax(axis=1))
})
sub_4f.to_csv("submission_clean_4family_blend.csv", index=False)

# Save all Day 1 clean OOF/test arrays for tomorrow's Large-model Grand Ensemble
np.savez(
    "clean_day1_all_probs.npz",
    oof_rob=oof_rob, test_rob=test_rob,
    oof_deb=oof_deb_ok, test_deb=test_deb_ok,
    oof_bge=oof_bge, test_bge=test_bge,
    oof_lex=oof_lex, test_lex=test_lex
)

files.download("submission_clean_4family_blend.csv")
files.download("clean_day1_all_probs.npz")
print("\nSaved & downloaded submission_clean_4family_blend.csv and clean_day1_all_probs.npz!")
print("Test Distribution:\n", sub_4f["emotion"].value_counts())

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/779 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.34GB            

... (9 outputs left out)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Saved & downloaded submission_clean_4family_blend.csv and clean_day1_all_probs.npz!
Test Distribution:
 emotion
joy         396
anger       347
sadness     278
surprise    188
disgust     120
fear        111
Name: count, dtype: int64


### Day 2: RoBERTa-large, DeBERTa-v3-large, and 6-family ensemble

In [ ]:
import os
import re
import json
import time
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, AutoConfig, get_cosine_schedule_with_warmup
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, classification_report
from google.colab import files

# ==========================================
# 1. Reproducibility & Device
# ==========================================
SEED = 1024  # Fresh seed for Day 2 Large models to maximize ensemble diversity
def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ==========================================
# 2. Load Competition Data & Day 1 Saved Probabilities
# ==========================================
with open("final_posts.json", "r", encoding="utf-8") as f:
    raw_posts = json.load(f)

if isinstance(raw_posts, dict):
    if all(isinstance(v, str) for v in list(raw_posts.values())[:5]):
        records = [{"post_id": str(k), "text": v} for k, v in raw_posts.items()]
    else:
        first_val = next(iter(raw_posts.values()))
        raw_posts = first_val if isinstance(first_val, list) else list(raw_posts.values())

if isinstance(raw_posts, list):
    records = []
    for item in raw_posts:
        post = item["root"]["_source"]["post"] if "root" in item else item
        pid = post.get("post_id") or post.get("id")
        txt = post.get("text") or post.get("content") or ""
        records.append({"post_id": str(pid), "text": txt})

posts_df = pd.DataFrame(records)
ident_df = pd.read_csv("data_identification.csv", dtype=str)
emo_df   = pd.read_csv("emotion.csv", dtype=str)
full_df  = posts_df.merge(ident_df, on="post_id", how="inner")

def light_clean(text):
    text = str(text)
    text = re.sub(r"http\S+|www\.\S+", "http", text)
    text = re.sub(r"@\w+", "@user", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

full_df["clean_text"] = full_df["text"].apply(light_clean)
train_df = full_df[full_df["set"] == "train"].merge(emo_df, on="post_id", how="inner").reset_index(drop=True)
test_df  = full_df[full_df["set"] == "test"].reset_index(drop=True)

sample_sub = pd.read_csv("sampleSubmission.csv", dtype=str)
sub_id_col = sample_sub.columns[0]
test_df = sample_sub[[sub_id_col]].merge(test_df, left_on=sub_id_col, right_on="post_id", how="left")

le = LabelEncoder()
y_all = le.fit_transform(train_df["label"].astype(str))
classes = le.classes_
num_classes = len(classes)
counts = np.bincount(y_all)

# IMPORTANT: Keep exact same fold split (random_state=42) so all OOF arrays align 1-to-1!
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Load Day 1 clean probabilities
day1 = np.load("clean_day1_all_probs.npz")
oof_rob_base, test_rob_base = day1["oof_rob"], day1["test_rob"]
oof_deb_base, test_deb_base = day1["oof_deb"], day1["test_deb"]
oof_bge, test_bge           = day1["oof_bge"], day1["test_bge"]
oof_lex, test_lex           = day1["oof_lex"], day1["test_lex"]
print("Loaded Day 1 clean probabilities successfully!")

# ==========================================
# 3. Mean+Max Pooling + Multi-Sample Dropout + Layer-wise LR Decay (LLRD)
# ==========================================
class LargeEmotionClassifier(nn.Module):
    def __init__(self, model_name, num_classes):
        super().__init__()
        self.config = AutoConfig.from_pretrained(model_name)
        self.config.update({"hidden_dropout_prob": 0.1, "attention_probs_dropout_prob": 0.1})
        self.backbone = AutoModel.from_pretrained(model_name, config=self.config, dtype=torch.float32)
        hidden = self.config.hidden_size
        self.dropouts = nn.ModuleList([nn.Dropout(0.1 * i) for i in range(1, 6)])
        self.fc = nn.Linear(hidden * 2, num_classes)

    def forward(self, input_ids, attention_mask):
        out = self.backbone(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state.float()
        mask = attention_mask.unsqueeze(-1).float()
        mean_pool = (out * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)
        max_pool = out.masked_fill(mask == 0, -1e9).max(dim=1).values
        feat = torch.cat([mean_pool, max_pool], dim=-1)
        logits = sum(self.fc(drop(feat)) for drop in self.dropouts) / len(self.dropouts)
        return logits

def get_llrd_optimizer(model, base_lr=1.5e-5, head_lr=1e-4, weight_decay=0.01, decay=0.95):
    no_decay = ["bias", "LayerNorm.weight", "LayerNorm.bias"]
    num_layers = model.config.num_hidden_layers
    optimizer_grouped_parameters = []

    # Classifier head
    optimizer_grouped_parameters.append({
        "params": [p for n, p in model.fc.named_parameters() if not any(nd in n for nd in no_decay)],
        "weight_decay": weight_decay, "lr": head_lr
    })
    optimizer_grouped_parameters.append({
        "params": [p for n, p in model.fc.named_parameters() if any(nd in n for nd in no_decay)],
        "weight_decay": 0.0, "lr": head_lr
    })

    # Transformer layers (top to bottom)
    layers = model.backbone.encoder.layer
    for layer_idx in range(num_layers - 1, -1, -1):
        lr = base_lr * (decay ** (num_layers - 1 - layer_idx))
        optimizer_grouped_parameters.append({
            "params": [p for n, p in layers[layer_idx].named_parameters() if not any(nd in n for nd in no_decay)],
            "weight_decay": weight_decay, "lr": lr
        })
        optimizer_grouped_parameters.append({
            "params": [p for n, p in layers[layer_idx].named_parameters() if any(nd in n for nd in no_decay)],
            "weight_decay": 0.0, "lr": lr
        })

    # Embeddings & remaining backbone params
    emb_lr = base_lr * (decay ** num_layers)
    assigned = {id(p) for g in optimizer_grouped_parameters for p in g["params"]}
    rem_decay, rem_no_decay = [], []
    for n, p in model.backbone.named_parameters():
        if id(p) not in assigned and p.requires_grad:
            if any(nd in n for nd in no_decay):
                rem_no_decay.append(p)
            else:
                rem_decay.append(p)
    optimizer_grouped_parameters.append({"params": rem_decay, "weight_decay": weight_decay, "lr": emb_lr})
    optimizer_grouped_parameters.append({"params": rem_no_decay, "weight_decay": 0.0, "lr": emb_lr})
    return torch.optim.AdamW(optimizer_grouped_parameters)

def train_large_backbone(model_name, base_lr=1.5e-5, epochs=4, batch_size=16, grad_accum=2, max_len=144, use_fp32=False):
    print(f"\nTraining 5-Fold Clean Large Backbone: {model_name} (LLRD, zero emotion supervision)...")
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    class TextDS(Dataset):
        def __init__(self, texts, labels=None):
            self.enc = tokenizer(list(texts), truncation=True, max_length=max_len, padding=False)
            self.labels = list(labels) if labels is not None else None
        def __len__(self):
            return len(self.enc["input_ids"])
        def __getitem__(self, i):
            d = {"input_ids": self.enc["input_ids"][i], "attention_mask": self.enc["attention_mask"][i]}
            if self.labels is not None:
                d["labels"] = self.labels[i]
            return d

    def collate(batch):
        ml = max(len(x["input_ids"]) for x in batch)
        pad_id = tokenizer.pad_token_id or 0
        ids, masks, labs = [], [], []
        for x in batch:
            pl = ml - len(x["input_ids"])
            ids.append(x["input_ids"] + [pad_id] * pl)
            masks.append(x["attention_mask"] + [0] * pl)
            if "labels" in x:
                labs.append(x["labels"])
        out = {"input_ids": torch.tensor(ids, dtype=torch.long), "attention_mask": torch.tensor(masks, dtype=torch.long)}
        if labs:
            out["labels"] = torch.tensor(labs, dtype=torch.long)
        return out

    test_loader = DataLoader(TextDS(test_df["clean_text"]), batch_size=64, shuffle=False, collate_fn=collate)
    class_weights = torch.tensor([(len(y_all) / (num_classes * counts[c])) ** 0.45 for c in range(num_classes)], dtype=torch.float32, device=device)

    oof_p = np.zeros((len(train_df), num_classes), dtype=np.float32)
    test_p = np.zeros((len(test_df), num_classes), dtype=np.float32)

    use_bf16 = (not use_fp32) and torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    amp_dtype = torch.bfloat16 if use_bf16 else torch.float16
    use_amp = (not use_fp32) and (device.type == "cuda")

    for fold, (tr_idx, va_idx) in enumerate(skf.split(train_df, y_all), 1):
        t0 = time.time()
        seed_everything(SEED + fold * 17)
        tr_loader = DataLoader(TextDS(train_df.loc[tr_idx, "clean_text"], y_all[tr_idx]), batch_size=batch_size, shuffle=True, collate_fn=collate)
        va_loader = DataLoader(TextDS(train_df.loc[va_idx, "clean_text"], y_all[va_idx]), batch_size=64, shuffle=False, collate_fn=collate)

        model = LargeEmotionClassifier(model_name, num_classes).float().to(device)
        optimizer = get_llrd_optimizer(model, base_lr=base_lr, head_lr=base_lr * 5, weight_decay=0.01, decay=0.95)
        total_steps = (len(tr_loader) // grad_accum) * epochs
        scheduler = get_cosine_schedule_with_warmup(optimizer, int(0.1 * total_steps), total_steps)
        criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.05)
        scaler = torch.amp.GradScaler("cuda", enabled=(use_amp and not use_bf16))

        best_f1, best_va, best_state = -1.0, None, None
        for ep in range(1, epochs + 1):
            model.train()
            optimizer.zero_grad()
            for step, b in enumerate(tr_loader, 1):
                with torch.amp.autocast("cuda", dtype=amp_dtype, enabled=use_amp):
                    logits = model(b["input_ids"].to(device), b["attention_mask"].to(device))
                    loss = criterion(logits, b["labels"].to(device)) / grad_accum
                if scaler.is_enabled():
                    scaler.scale(loss).backward()
                else:
                    loss.backward()

                if step % grad_accum == 0 or step == len(tr_loader):
                    if scaler.is_enabled():
                        scaler.unscale_(optimizer)
                        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                        scaler.step(optimizer)
                        scaler.update()
                    else:
                        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                        optimizer.step()
                    scheduler.step()
                    optimizer.zero_grad()

            model.eval()
            va_list = []
            with torch.inference_mode():
                for b in va_loader:
                    with torch.amp.autocast("cuda", dtype=amp_dtype, enabled=use_amp):
                        logits = model(b["input_ids"].to(device), b["attention_mask"].to(device))
                        va_list.append(torch.softmax(logits.float(), dim=-1).cpu().numpy())
            va_probs = np.vstack(va_list)
            f1_ep = f1_score(y_all[va_idx], va_probs.argmax(axis=1), average="macro")
            if f1_ep > best_f1:
                best_f1, best_va = f1_ep, va_probs
                best_state = {k: v.cpu() for k, v in model.state_dict().items()}

        oof_p[va_idx] = best_va
        model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
        model.eval()
        te_list = []
        with torch.inference_mode():
            for b in test_loader:
                with torch.amp.autocast("cuda", dtype=amp_dtype, enabled=use_amp):
                    logits = model(b["input_ids"].to(device), b["attention_mask"].to(device))
                    te_list.append(torch.softmax(logits.float(), dim=-1).cpu().numpy())
        test_p += np.vstack(te_list) / 5.0
        print(f"  Fold {fold}/5 | Val Macro-F1: {best_f1:.4f} | {time.time() - t0:.1f}s")

    cv_f1 = f1_score(y_all, oof_p.argmax(axis=1), average="macro")
    print(f"--> {model_name} 5-Fold OOF Macro-F1: {cv_f1:.4f}")
    return oof_p, test_p

# Train both clean Large backbones (RoBERTa-large uses fast bf16/fp16; DeBERTa-v3-large uses safe FP32)
oof_rob_large, test_rob_large = train_large_backbone("FacebookAI/roberta-large", base_lr=1.5e-5, epochs=4, batch_size=16, grad_accum=2, use_fp32=False)
oof_deb_large, test_deb_large = train_large_backbone("microsoft/deberta-v3-large", base_lr=1.2e-5, epochs=4, batch_size=16, grad_accum=2, use_fp32=True)

# ==========================================
# 4. 6-Family Grand Ensemble (Arithmetic, Geometric & OOF-Weighted)
# ==========================================
# 6 completely compliant clean models:
# [0] RoBERTa-large, [1] DeBERTa-v3-large, [2] RoBERTa-base, [3] DeBERTa-v3-base, [4] BGE-Large, [5] Lexical TF-IDF
oofs  = [oof_rob_large, oof_deb_large, oof_rob_base, oof_deb_base, oof_bge, oof_lex]
tests = [test_rob_large, test_deb_large, test_rob_base, test_deb_base, test_bge, test_lex]
names = ["RoBERTa-large", "DeBERTa-v3-large", "RoBERTa-base", "DeBERTa-v3-base", "BGE-Large", "Lexical-TFIDF"]

print("\n" + "=" * 76)
print("INDIVIDUAL CLEAN MODEL 5-FOLD OOF MACRO-F1 SUMMARY")
print("=" * 76)
for n, o in zip(names, oofs):
    print(f"  {n:<20s}: {f1_score(y_all, o.argmax(axis=1), average='macro'):.4f}")

# Candidate A: Balanced 6-Family Arithmetic Blend (Large-heavy)
w_arith = np.array([0.30, 0.25, 0.15, 0.10, 0.12, 0.08])
oof_arith = sum(w * o for w, o in zip(w_arith, oofs))
test_arith = sum(w * t for w, t in zip(w_arith, tests))
f1_arith = f1_score(y_all, oof_arith.argmax(axis=1), average="macro")

# Candidate B: 6-Family Geometric Mean Blend (Log-probability average, strong against overconfident outliers)
eps = 1e-7
oof_geom = np.exp(sum(w * np.log(np.clip(o, eps, 1.0)) for w, o in zip(w_arith, oofs)))
test_geom = np.exp(sum(w * np.log(np.clip(t, eps, 1.0)) for w, t in zip(w_arith, tests)))
f1_geom = f1_score(y_all, oof_geom.argmax(axis=1), average="macro")

# Candidate C: Random Dirichlet OOF Search (3,000 weight combinations, pure raw argmax)
rng = np.random.default_rng(42)
best_dir_w, best_dir_f1 = w_arith, f1_arith
for _ in range(3000):
    w = rng.dirichlet([3.0, 2.5, 1.5, 1.0, 1.2, 0.8])
    s = f1_score(y_all, sum(wi * oi for wi, oi in zip(w, oofs)).argmax(axis=1), average="macro")
    if s > best_dir_f1:
        best_dir_f1, best_dir_w = s, w

oof_dir = sum(w * o for w, o in zip(best_dir_w, oofs))
test_dir = sum(w * t for w, t in zip(best_dir_w, tests))

print("=" * 76)
print("DAY 2 GRAND ENSEMBLE RESULTS (100% COMPLIANT, RAW ARGMAX)")
print("=" * 76)
print(f"  1. RoBERTa-large Single 5-Fold         : {f1_score(y_all, oof_rob_large.argmax(axis=1), average='macro'):.4f}")
print(f"  2. DeBERTa-v3-large Single 5-Fold      : {f1_score(y_all, oof_deb_large.argmax(axis=1), average='macro'):.4f}")
print(f"  3. 6-Family Arithmetic Blend           : {f1_arith:.4f}")
print(f"  4. 6-Family Geometric Mean Blend       : {f1_geom:.4f}")
print(f"  5. 6-Family Dirichlet OOF Optimal      : {best_dir_f1:.4f} | Weights: {np.round(best_dir_w, 3)}")
print("=" * 76)

# Save all 5 Day 2 submission candidates + full probability archive
subs = {
    "sub_day2_1_roberta_large.csv": test_rob_large,
    "sub_day2_2_deberta_large.csv": test_deb_large,
    "sub_day2_3_6family_arith.csv": test_arith,
    "sub_day2_4_6family_geom.csv": test_geom,
    "sub_day2_5_6family_dirichlet.csv": test_dir,
}
for fname, probs in subs.items():
    df_out = pd.DataFrame({"id": test_df[sub_id_col], "emotion": le.inverse_transform(probs.argmax(axis=1))})
    df_out.to_csv(fname, index=False)
    files.download(fname)

np.savez(
    "clean_day2_all_probs.npz",
    oof_rob_large=oof_rob_large, test_rob_large=test_rob_large,
    oof_deb_large=oof_deb_large, test_deb_large=test_deb_large,
    oof_rob_base=oof_rob_base, test_rob_base=test_rob_base,
    oof_deb_base=oof_deb_base, test_deb_base=test_deb_base,
    oof_bge=oof_bge, test_bge=test_bge,
    oof_lex=oof_lex, test_lex=test_lex
)
files.download("clean_day2_all_probs.npz")

Using device: cuda
Loaded Day 1 clean probabilities successfully!

Training 5-Fold Clean Large Backbone: FacebookAI/roberta-large (LLRD, zero emotion supervision)...


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

... (49 outputs left out)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Day 2 (extra): 5-family blend without DeBERTa-v3-large

In [ ]:
import os
import numpy as np
import pandas as pd
from google.colab import files

# 若檔案已不在暫存區，觸發上傳 clean_day2_all_probs.npz 與 sampleSubmission.csv
needed = [f for f in ["clean_day2_all_probs.npz", "sampleSubmission.csv"] if not os.path.exists(f)]
if needed:
    print("請選擇並上傳:", needed)
    files.upload()

d = np.load("clean_day2_all_probs.npz")
sample_sub = pd.read_csv("sampleSubmission.csv", dtype=str)
sub_id_col = sample_sub.columns[0]
classes = np.array(["anger", "disgust", "fear", "joy", "sadness", "surprise"])

# 5-Family Clean Blend (踢掉欠擬合的 DeBERTa-v3-large)
test_5f = (
    0.35 * d["test_rob_large"]
    + 0.25 * d["test_rob_base"]
    + 0.18 * d["test_bge"]
    + 0.12 * d["test_deb_base"]
    + 0.10 * d["test_lex"]
)

sub_5f = pd.DataFrame({
    "id": sample_sub[sub_id_col],
    "emotion": classes[test_5f.argmax(axis=1)]
})
sub_5f.to_csv("sub_day2_bonus_5family_clean.csv", index=False)
print("Preview:\n", sub_5f.head(3))
print("\nTest Distribution:\n", sub_5f["emotion"].value_counts())
files.download("sub_day2_bonus_5family_clean.csv")

請選擇並上傳: ['clean_day2_all_probs.npz', 'sampleSubmission.csv']


Saving sampleSubmission.csv to sampleSubmission.csv
Saving clean_day2_all_probs.npz to clean_day2_all_probs.npz
Preview:
         id   emotion
0  xyzx9jp  surprise
1  l3vzzxt   sadness
2  g680iug   sadness

Test Distribution:
 emotion
joy         394
anger       342
sadness     273
surprise    186
disgust     129
fear        116
Name: count, dtype: int64


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Day 3: E5-large-v2 embeddings and 7-family ensemble

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving clean_day2_all_probs.npz to clean_day2_all_probs.npz
Saving data_identification.csv to data_identification.csv
Saving emotion.csv to emotion.csv
Saving final_posts.json to final_posts.json
Saving sampleSubmission.csv to sampleSubmission.csv


In [ ]:
!pip install -q sentence-transformers

import os
import re
import json
import time
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_cosine_schedule_with_warmup
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder, normalize
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import f1_score
from google.colab import files

# ==========================================
# 1. Reproducibility & Device
# ==========================================
SEED = 2026
def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
assert device.type == "cuda", "請先切換至 GPU (L4 或 A100) 再執行！"

# ==========================================
# 2. Load Competition Data & Day 2 Saved Probabilities
# ==========================================
with open("final_posts.json", "r", encoding="utf-8") as f:
    raw_posts = json.load(f)

if isinstance(raw_posts, dict):
    if all(isinstance(v, str) for v in list(raw_posts.values())[:5]):
        records = [{"post_id": str(k), "text": v} for k, v in raw_posts.items()]
    else:
        first_val = next(iter(raw_posts.values()))
        raw_posts = first_val if isinstance(first_val, list) else list(raw_posts.values())

if isinstance(raw_posts, list):
    records = []
    for item in raw_posts:
        post = item["root"]["_source"]["post"] if "root" in item else item
        pid = post.get("post_id") or post.get("id")
        txt = post.get("text") or post.get("content") or ""
        records.append({"post_id": str(pid), "text": txt})

posts_df = pd.DataFrame(records)
ident_df = pd.read_csv("data_identification.csv", dtype=str)
emo_df   = pd.read_csv("emotion.csv", dtype=str)
full_df  = posts_df.merge(ident_df, on="post_id", how="inner")

def light_clean(text):
    text = str(text)
    text = re.sub(r"http\S+|www\.\S+", "http", text)
    text = re.sub(r"@\w+", "@user", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

full_df["clean_text"] = full_df["text"].apply(light_clean)
train_df = full_df[full_df["set"] == "train"].merge(emo_df, on="post_id", how="inner").reset_index(drop=True)
test_df  = full_df[full_df["set"] == "test"].reset_index(drop=True)

sample_sub = pd.read_csv("sampleSubmission.csv", dtype=str)
sub_id_col = sample_sub.columns[0]
test_df = sample_sub[[sub_id_col]].merge(test_df, left_on=sub_id_col, right_on="post_id", how="left")

le = LabelEncoder()
y_all = le.fit_transform(train_df["label"].astype(str))
classes = le.classes_
num_classes = len(classes)
counts = np.bincount(y_all)

# Keep exact same 5-fold split (random_state=42) for 1-to-1 OOF alignment
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Load Day 2 probabilities
d2 = np.load("clean_day2_all_probs.npz")
oof_rob_large, test_rob_large = d2["oof_rob_large"], d2["test_rob_large"]
oof_deb_large_old, test_deb_large_old = d2["oof_deb_large"], d2["test_deb_large"]
oof_rob_base, test_rob_base   = d2["oof_rob_base"], d2["test_rob_base"]
oof_deb_base, test_deb_base   = d2["oof_deb_base"], d2["test_deb_base"]
oof_bge, test_bge             = d2["oof_bge"], d2["test_bge"]
oof_lex, test_lex             = d2["oof_lex"], d2["test_lex"]
print("Loaded Day 2 clean probabilities successfully!")

# ==========================================
# 3. Model Family #7: Clean intfloat/e5-large-v2 (MIT, zero emotion supervision)
# ==========================================
print("\n[1/2] Extracting Clean E5-Large-v2 Embeddings (zero emotion supervision)...")
e5_model = SentenceTransformer("intfloat/e5-large-v2", device=str(device))
# E5 expects "passage: " prefix for document/post encoding
tr_e5_texts = ["passage: " + t for t in train_df["clean_text"].tolist()]
te_e5_texts = ["passage: " + t for t in test_df["clean_text"].tolist()]

X_tr_e5 = normalize(e5_model.encode(tr_e5_texts, batch_size=64, show_progress_bar=False))
X_te_e5 = normalize(e5_model.encode(te_e5_texts, batch_size=64, show_progress_bar=False))

tempered_w = {c: (len(y_all) / (num_classes * counts[c])) ** 0.45 for c in range(num_classes)}
oof_e5 = np.zeros((len(train_df), num_classes), dtype=np.float32)
test_e5 = np.zeros((len(test_df), num_classes), dtype=np.float32)

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_tr_e5, y_all)):
    m_svc = CalibratedClassifierCV(LinearSVC(C=0.35, class_weight=tempered_w, random_state=SEED + fold), cv=3)
    m_lr  = LogisticRegression(C=2.0, class_weight=tempered_w, max_iter=1000, random_state=SEED + fold)
    m_svc.fit(X_tr_e5[tr_idx], y_all[tr_idx])
    m_lr.fit(X_tr_e5[tr_idx], y_all[tr_idx])
    oof_e5[va_idx] = 0.5 * m_svc.predict_proba(X_tr_e5[va_idx]) + 0.5 * m_lr.predict_proba(X_tr_e5[va_idx])
    test_e5 += (0.5 * m_svc.predict_proba(X_te_e5) + 0.5 * m_lr.predict_proba(X_te_e5)) / 5.0

print(f"--> Clean E5-Large-v2 5-Fold OOF Macro-F1: {f1_score(y_all, oof_e5.argmax(axis=1), average='macro'):.4f}")
del e5_model
torch.cuda.empty_cache()

# ==========================================
# 4. Fixed DeBERTa-v3-large (Standard Classification Head, Uniform LR=1.5e-5, Safe FP32)
# ==========================================
def train_fixed_deberta_large(model_name="microsoft/deberta-v3-large", lr=1.5e-5, epochs=4, batch_size=16, grad_accum=2, max_len=128):
    print(f"\n[2/2] Training Fixed 5-Fold {model_name} (Standard Head, Uniform LR={lr}, FP32)...")
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    class TextDS(Dataset):
        def __init__(self, texts, labels=None):
            self.enc = tokenizer(list(texts), truncation=True, max_length=max_len, padding=False)
            self.labels = list(labels) if labels is not None else None
        def __len__(self):
            return len(self.enc["input_ids"])
        def __getitem__(self, i):
            d = {"input_ids": self.enc["input_ids"][i], "attention_mask": self.enc["attention_mask"][i]}
            if self.labels is not None:
                d["labels"] = self.labels[i]
            return d

    def collate(batch):
        ml = max(len(x["input_ids"]) for x in batch)
        pad_id = tokenizer.pad_token_id or 0
        ids, masks, labs = [], [], []
        for x in batch:
            pl = ml - len(x["input_ids"])
            ids.append(x["input_ids"] + [pad_id] * pl)
            masks.append(x["attention_mask"] + [0] * pl)
            if "labels" in x:
                labs.append(x["labels"])
        out = {"input_ids": torch.tensor(ids, dtype=torch.long), "attention_mask": torch.tensor(masks, dtype=torch.long)}
        if labs:
            out["labels"] = torch.tensor(labs, dtype=torch.long)
        return out

    test_loader = DataLoader(TextDS(test_df["clean_text"]), batch_size=64, shuffle=False, collate_fn=collate)
    class_weights = torch.tensor([(len(y_all) / (num_classes * counts[c])) ** 0.45 for c in range(num_classes)], dtype=torch.float32, device=device)

    oof_p = np.zeros((len(train_df), num_classes), dtype=np.float32)
    test_p = np.zeros((len(test_df), num_classes), dtype=np.float32)

    for fold, (tr_idx, va_idx) in enumerate(skf.split(train_df, y_all), 1):
        t0 = time.time()
        seed_everything(SEED + fold * 13)
        tr_loader = DataLoader(TextDS(train_df.loc[tr_idx, "clean_text"], y_all[tr_idx]), batch_size=batch_size, shuffle=True, collate_fn=collate)
        va_loader = DataLoader(TextDS(train_df.loc[va_idx, "clean_text"], y_all[va_idx]), batch_size=64, shuffle=False, collate_fn=collate)

        model = AutoModelForSequenceClassification.from_pretrained(
            model_name, num_labels=num_classes, hidden_dropout_prob=0.1, attention_probs_dropout_prob=0.1, dtype=torch.float32
        ).float().to(device)

        optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
        total_steps = (len(tr_loader) // grad_accum) * epochs
        scheduler = get_cosine_schedule_with_warmup(optimizer, int(0.1 * total_steps), total_steps)
        criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.05)

        best_f1, best_va, best_state = -1.0, None, None
        for ep in range(1, epochs + 1):
            model.train()
            optimizer.zero_grad()
            for step, b in enumerate(tr_loader, 1):
                logits = model(input_ids=b["input_ids"].to(device), attention_mask=b["attention_mask"].to(device)).logits.float()
                loss = criterion(logits, b["labels"].to(device)) / grad_accum
                loss.backward()
                if step % grad_accum == 0 or step == len(tr_loader):
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    optimizer.step()
                    scheduler.step()
                    optimizer.zero_grad()

            model.eval()
            va_list = []
            with torch.inference_mode():
                for b in va_loader:
                    logits = model(input_ids=b["input_ids"].to(device), attention_mask=b["attention_mask"].to(device)).logits.float()
                    va_list.append(torch.softmax(logits, dim=-1).cpu().numpy())
            va_probs = np.vstack(va_list)
            f1_ep = f1_score(y_all[va_idx], va_probs.argmax(axis=1), average="macro")
            if f1_ep > best_f1:
                best_f1, best_va = f1_ep, va_probs
                best_state = {k: v.cpu() for k, v in model.state_dict().items()}

        oof_p[va_idx] = best_va
        model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
        model.eval()
        te_list = []
        with torch.inference_mode():
            for b in test_loader:
                logits = model(input_ids=b["input_ids"].to(device), attention_mask=b["attention_mask"].to(device)).logits.float()
                te_list.append(torch.softmax(logits, dim=-1).cpu().numpy())
        test_p += np.vstack(te_list) / 5.0
        print(f"  Fold {fold}/5 | Val Macro-F1: {best_f1:.4f} | {time.time() - t0:.1f}s")

    print(f"--> Fixed {model_name} 5-Fold OOF Macro-F1: {f1_score(y_all, oof_p.argmax(axis=1), average='macro'):.4f}")
    return oof_p, test_p

oof_deb_large_fix, test_deb_large_fix = train_fixed_deberta_large()

# ==========================================
# 5. Day 3 7-Family Grand Ensemble (No Overfitting!)
# ==========================================
# [0] RoBERTa-large, [1] DeBERTa-v3-large (Fixed), [2] RoBERTa-base,
# [3] DeBERTa-v3-base, [4] BGE-Large, [5] E5-Large-v2, [6] Lexical-TFIDF
oofs  = [oof_rob_large, oof_deb_large_fix, oof_rob_base, oof_deb_base, oof_bge, oof_e5, oof_lex]
tests = [test_rob_large, test_deb_large_fix, test_rob_base, test_deb_base, test_bge, test_e5, test_lex]
names = ["RoBERTa-large", "DeBERTa-v3-large(Fix)", "RoBERTa-base", "DeBERTa-v3-base", "BGE-Large", "E5-Large-v2", "Lexical-TFIDF"]

print("\n" + "=" * 78)
print("DAY 3 INDIVIDUAL CLEAN MODEL 5-FOLD OOF MACRO-F1 SUMMARY")
print("=" * 78)
for n, o in zip(names, oofs):
    print(f"  {n:<24s}: {f1_score(y_all, o.argmax(axis=1), average='macro'):.4f}")

# Candidate 2: Balanced 7-Family Arithmetic Blend
w_bal = np.array([0.24, 0.24, 0.14, 0.08, 0.11, 0.11, 0.08])
oof_bal = sum(w * o for w, o in zip(w_bal, oofs))
test_bal = sum(w * t for w, t in zip(w_bal, tests))

# Candidate 3: Large-Heavy 7-Family Arithmetic Blend (RoBERTa-L + DeB-L = 56%)
w_lrg = np.array([0.28, 0.28, 0.12, 0.06, 0.10, 0.10, 0.06])
oof_lrg = sum(w * o for w, o in zip(w_lrg, oofs))
test_lrg = sum(w * t for w, t in zip(w_lrg, tests))

# Candidate 4: 7-Family Geometric Mean Blend (Log-Probability Average)
eps = 1e-7
oof_geom = np.exp(sum(w * np.log(np.clip(o, eps, 1.0)) for w, o in zip(w_bal, oofs)))
test_geom = np.exp(sum(w * np.log(np.clip(t, eps, 1.0)) for w, t in zip(w_bal, tests)))

# Candidate 5: Strongly Regularized L2 LogisticRegression Meta-Stacker (Strict 5-Fold Nested OOF)
X_meta_tr = np.hstack(oofs)
X_meta_te = np.hstack(tests)
oof_meta = np.zeros((len(train_df), num_classes), dtype=np.float32)
test_meta = np.zeros((len(test_df), num_classes), dtype=np.float32)

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_meta_tr, y_all)):
    meta_clf = LogisticRegression(C=0.1, max_iter=1000, random_state=SEED + fold)
    meta_clf.fit(X_meta_tr[tr_idx], y_all[tr_idx])
    oof_meta[va_idx] = meta_clf.predict_proba(X_meta_tr[va_idx])
    test_meta += meta_clf.predict_proba(X_meta_te) / 5.0

print("=" * 78)
print("DAY 3 GRAND ENSEMBLE RESULTS (100% COMPLIANT, ZERO OVERFITTING)")
print("=" * 78)
print(f"  1. Fixed DeBERTa-v3-large Single 5-Fold : {f1_score(y_all, oof_deb_large_fix.argmax(axis=1), average='macro'):.4f}")
print(f"  2. 7-Family Balanced Arithmetic Blend   : {f1_score(y_all, oof_bal.argmax(axis=1), average='macro'):.4f}")
print(f"  3. 7-Family Large-Heavy Arith Blend     : {f1_score(y_all, oof_lrg.argmax(axis=1), average='macro'):.4f}")
print(f"  4. 7-Family Geometric Mean Blend        : {f1_score(y_all, oof_geom.argmax(axis=1), average='macro'):.4f}")
print(f"  5. 7-Family L2 Meta-Stacker (C=0.1)     : {f1_score(y_all, oof_meta.argmax(axis=1), average='macro'):.4f}")
print("=" * 78)

subs = {
    "sub_day3_1_deberta_large_fixed.csv": test_deb_large_fix,
    "sub_day3_2_7family_arith_balanced.csv": test_bal,
    "sub_day3_3_7family_large_heavy.csv": test_lrg,
    "sub_day3_4_7family_geom.csv": test_geom,
    "sub_day3_5_7family_meta_l2.csv": test_meta,
}
for fname, probs in subs.items():
    df_out = pd.DataFrame({"id": test_df[sub_id_col], "emotion": le.inverse_transform(probs.argmax(axis=1))})
    df_out.to_csv(fname, index=False)
    files.download(fname)

np.savez(
    "clean_day3_all_probs.npz",
    oof_rob_large=oof_rob_large, test_rob_large=test_rob_large,
    oof_deb_large_fix=oof_deb_large_fix, test_deb_large_fix=test_deb_large_fix,
    oof_deb_large_old=oof_deb_large_old, test_deb_large_old=test_deb_large_old,
    oof_rob_base=oof_rob_base, test_rob_base=test_rob_base,
    oof_deb_base=oof_deb_base, test_deb_base=test_deb_base,
    oof_bge=oof_bge, test_bge=test_bge,
    oof_e5=oof_e5, test_e5=test_e5,
    oof_lex=oof_lex, test_lex=test_lex
)
files.download("clean_day3_all_probs.npz")

Using device: cuda
Loaded Day 2 clean probabilities successfully!

[1/2] Extracting Clean E5-Large-v2 Embeddings (zero emotion supervision)...


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/67.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.34GB            

... (40 outputs left out)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from google.colab import files

# 改用五折全正常的 Day 2 DeBERTa-v3-large + 新加入的 E5-Large-v2 (7-Family 完整健康版)
oofs_ok  = [oof_rob_large, oof_deb_large_old, oof_rob_base, oof_deb_base, oof_bge, oof_e5, oof_lex]
tests_ok = [test_rob_large, test_deb_large_old, test_rob_base, test_deb_base, test_bge, test_e5, test_lex]

# 1. 7-Family Arithmetic Blend (沿用昨天 0.6127 的黃金比例，將 BGE 拆成 BGE + E5)
w_arith_v1 = np.array([0.28, 0.22, 0.14, 0.08, 0.10, 0.10, 0.08])
oof_v1 = sum(w * o for w, o in zip(w_arith_v1, oofs_ok))
test_v1 = sum(w * t for w, t in zip(w_arith_v1, tests_ok))

# 2. 7-Family RoBERTa-Heavy Arithmetic Blend (加重最強單模 RoBERTa-L 32%)
w_arith_v2 = np.array([0.32, 0.20, 0.14, 0.08, 0.10, 0.10, 0.06])
oof_v2 = sum(w * o for w, o in zip(w_arith_v2, oofs_ok))
test_v2 = sum(w * t for w, t in zip(w_arith_v2, tests_ok))

# 3. 7-Family Geometric Mean Blend (Log-Probability Average)
eps = 1e-7
oof_v3 = np.exp(sum(w * np.log(np.clip(o, eps, 1.0)) for w, o in zip(w_arith_v1, oofs_ok)))
test_v3 = np.exp(sum(w * np.log(np.clip(t, eps, 1.0)) for w, t in zip(w_arith_v1, tests_ok)))

# 4 & 5. 7-Family L2 Meta-Stacker (C=0.1 與更強正則化 C=0.05，嚴格 5-Fold Nested OOF)
X_tr_ok = np.hstack(oofs_ok)
X_te_ok = np.hstack(tests_ok)

def run_meta(C_val):
    oof_m = np.zeros((len(train_df), num_classes), dtype=np.float32)
    test_m = np.zeros((len(test_df), num_classes), dtype=np.float32)
    for fold, (tr_idx, va_idx) in enumerate(skf.split(X_tr_ok, y_all)):
        clf = LogisticRegression(C=C_val, max_iter=1000, random_state=42 + fold)
        clf.fit(X_tr_ok[tr_idx], y_all[tr_idx])
        oof_m[va_idx] = clf.predict_proba(X_tr_ok[va_idx])
        test_m += clf.predict_proba(X_te_ok) / 5.0
    return oof_m, test_m

oof_v4, test_v4 = run_meta(0.10)
oof_v5, test_v5 = run_meta(0.05)

print("=" * 76)
print("HEALTHY 7-FAMILY DAY 3 ENSEMBLE SUMMARY (100% COMPLIANT)")
print("=" * 76)
print(f"  1. 7-Family Arith Balanced (v1)   : {f1_score(y_all, oof_v1.argmax(axis=1), average='macro'):.4f}")
print(f"  2. 7-Family Arith RoBERTa-Heavy   : {f1_score(y_all, oof_v2.argmax(axis=1), average='macro'):.4f}")
print(f"  3. 7-Family Geometric Mean        : {f1_score(y_all, oof_v3.argmax(axis=1), average='macro'):.4f}")
print(f"  4. 7-Family L2 Meta-Stacker C=0.1 : {f1_score(y_all, oof_v4.argmax(axis=1), average='macro'):.4f}")
print(f"  5. 7-Family L2 Meta-Stacker C=0.05: {f1_score(y_all, oof_v5.argmax(axis=1), average='macro'):.4f}")
print("=" * 76)

subs_day3_ok = {
    "sub_day3_ok_1_7fam_arith_bal.csv": test_v1,
    "sub_day3_ok_2_7fam_arith_robheavy.csv": test_v2,
    "sub_day3_ok_3_7fam_geom.csv": test_v3,
    "sub_day3_ok_4_7fam_meta_c010.csv": test_v4,
    "sub_day3_ok_5_7fam_meta_c005.csv": test_v5,
}
for fname, probs in subs_day3_ok.items():
    pd.DataFrame({"id": test_df[sub_id_col], "emotion": le.inverse_transform(probs.argmax(axis=1))}).to_csv(fname, index=False)
    files.download(fname)

HEALTHY 7-FAMILY DAY 3 ENSEMBLE SUMMARY (100% COMPLIANT)
  1. 7-Family Arith Balanced (v1)   : 0.6175
  2. 7-Family Arith RoBERTa-Heavy   : 0.6155
  3. 7-Family Geometric Mean        : 0.6190
  4. 7-Family L2 Meta-Stacker C=0.1 : 0.6190
  5. 7-Family L2 Meta-Stacker C=0.05: 0.6182


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Day 4 Final Version: RoBERTa (Large, Seed 777), MPNet, and 9-family — generating the two final selected submissions.

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving clean_day3_all_probs.npz to clean_day3_all_probs.npz
Saving data_identification.csv to data_identification.csv
Saving emotion.csv to emotion.csv
Saving final_posts.json to final_posts.json
Saving sampleSubmission.csv to sampleSubmission.csv


In [ ]:
!pip install -q sentence-transformers

import os
import re
import json
import time
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, AutoConfig, get_cosine_schedule_with_warmup
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder, normalize
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import f1_score
from google.colab import files

# ==========================================
# 1. Reproducibility & Device
# ==========================================
SEED = 777  # Second independent seed for RoBERTa-large diversity
def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
assert device.type == "cuda", "請先切換至 GPU (L4 或 A100) 再執行！"

# ==========================================
# 2. Load Competition Data & Day 3 Saved Probabilities
# ==========================================
with open("final_posts.json", "r", encoding="utf-8") as f:
    raw_posts = json.load(f)

if isinstance(raw_posts, dict):
    if all(isinstance(v, str) for v in list(raw_posts.values())[:5]):
        records = [{"post_id": str(k), "text": v} for k, v in raw_posts.items()]
    else:
        first_val = next(iter(raw_posts.values()))
        raw_posts = first_val if isinstance(first_val, list) else list(raw_posts.values())

if isinstance(raw_posts, list):
    records = []
    for item in raw_posts:
        post = item["root"]["_source"]["post"] if "root" in item else item
        pid = post.get("post_id") or post.get("id")
        txt = post.get("text") or post.get("content") or ""
        records.append({"post_id": str(pid), "text": txt})

posts_df = pd.DataFrame(records)
ident_df = pd.read_csv("data_identification.csv", dtype=str)
emo_df   = pd.read_csv("emotion.csv", dtype=str)
full_df  = posts_df.merge(ident_df, on="post_id", how="inner")

def light_clean(text):
    text = str(text)
    text = re.sub(r"http\S+|www\.\S+", "http", text)
    text = re.sub(r"@\w+", "@user", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

full_df["clean_text"] = full_df["text"].apply(light_clean)
train_df = full_df[full_df["set"] == "train"].merge(emo_df, on="post_id", how="inner").reset_index(drop=True)
test_df  = full_df[full_df["set"] == "test"].reset_index(drop=True)

sample_sub = pd.read_csv("sampleSubmission.csv", dtype=str)
sub_id_col = sample_sub.columns[0]
test_df = sample_sub[[sub_id_col]].merge(test_df, left_on=sub_id_col, right_on="post_id", how="left")

le = LabelEncoder()
y_all = le.fit_transform(train_df["label"].astype(str))
classes = le.classes_
num_classes = len(classes)
counts = np.bincount(y_all)

# Exact same 5-fold split (random_state=42) for 1-to-1 OOF alignment
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Load Day 3 probabilities (7 healthy families)
d3 = np.load("clean_day3_all_probs.npz")
oof_rob_l1, test_rob_l1 = d3["oof_rob_large"], d3["test_rob_large"]
oof_deb_l,  test_deb_l  = d3["oof_deb_large_old"], d3["test_deb_large_old"]
oof_rob_b,  test_rob_b  = d3["oof_rob_base"], d3["test_rob_base"]
oof_deb_b,  test_deb_b  = d3["oof_deb_base"], d3["test_deb_base"]
oof_bge,    test_bge    = d3["oof_bge"], d3["test_bge"]
oof_e5,     test_e5     = d3["oof_e5"], d3["test_e5"]
oof_lex,    test_lex    = d3["oof_lex"], d3["test_lex"]
print("Loaded Day 3 clean probabilities successfully!")

# ==========================================
# 3. Model Family #8: Clean all-mpnet-base-v2 (Apache 2.0, zero emotion supervision)
# ==========================================
print("\n[1/2] Extracting Clean all-mpnet-base-v2 Embeddings (zero emotion supervision)...")
mpnet = SentenceTransformer("sentence-transformers/all-mpnet-base-v2", device=str(device))
X_tr_mp = normalize(mpnet.encode(train_df["clean_text"].tolist(), batch_size=64, show_progress_bar=False))
X_te_mp = normalize(mpnet.encode(test_df["clean_text"].tolist(), batch_size=64, show_progress_bar=False))

tempered_w = {c: (len(y_all) / (num_classes * counts[c])) ** 0.45 for c in range(num_classes)}
oof_mpnet = np.zeros((len(train_df), num_classes), dtype=np.float32)
test_mpnet = np.zeros((len(test_df), num_classes), dtype=np.float32)

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_tr_mp, y_all)):
    m_svc = CalibratedClassifierCV(LinearSVC(C=0.35, class_weight=tempered_w, random_state=SEED + fold), cv=3)
    m_lr  = LogisticRegression(C=2.0, class_weight=tempered_w, max_iter=1000, random_state=SEED + fold)
    m_svc.fit(X_tr_mp[tr_idx], y_all[tr_idx])
    m_lr.fit(X_tr_mp[tr_idx], y_all[tr_idx])
    oof_mpnet[va_idx] = 0.5 * m_svc.predict_proba(X_tr_mp[va_idx]) + 0.5 * m_lr.predict_proba(X_tr_mp[va_idx])
    test_mpnet += (0.5 * m_svc.predict_proba(X_te_mp) + 0.5 * m_lr.predict_proba(X_te_mp)) / 5.0

print(f"--> Clean all-mpnet-base-v2 5-Fold OOF Macro-F1: {f1_score(y_all, oof_mpnet.argmax(axis=1), average='macro'):.4f}")
del mpnet
torch.cuda.empty_cache()

# ==========================================
# 4. Model Family #9: Clean RoBERTa-large Seed 2 (LLRD + Mean/Max Pool + 5x MSD)
# ==========================================
class MeanMaxClassifier(nn.Module):
    def __init__(self, model_name, num_classes):
        super().__init__()
        self.config = AutoConfig.from_pretrained(model_name)
        self.config.update({"hidden_dropout_prob": 0.1, "attention_probs_dropout_prob": 0.1})
        self.backbone = AutoModel.from_pretrained(model_name, config=self.config)
        self.dropouts = nn.ModuleList([nn.Dropout(0.1 * i) for i in range(1, 6)])
        self.fc = nn.Linear(self.config.hidden_size * 2, num_classes)

    def forward(self, input_ids, attention_mask):
        out = self.backbone(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        mask = attention_mask.unsqueeze(-1).float()
        mean_pool = (out * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)
        max_pool  = out.masked_fill(mask == 0, -1e4).max(dim=1).values
        feat = torch.cat([mean_pool, max_pool], dim=-1)
        return sum(self.fc(d(feat)) for d in self.dropouts) / len(self.dropouts)

def get_llrd_params(model, base_lr=2e-5, head_lr=1e-4, llrd=0.95, wd=0.01):
    no_decay = ["bias", "LayerNorm.bias", "LayerNorm.weight"]
    groups = [{"params": [p for n, p in model.fc.named_parameters()], "lr": head_lr, "weight_decay": wd}]
    num_layers = model.config.num_hidden_layers
    for name, p in model.backbone.named_parameters():
        if not p.requires_grad:
            continue
        decay = 0.0 if any(nd in name for nd in no_decay) else wd
        if "embeddings" in name:
            lr = base_lr * (llrd ** (num_layers + 1))
        elif "encoder.layer." in name:
            layer_idx = int(name.split("encoder.layer.")[1].split(".")[0])
            lr = base_lr * (llrd ** (num_layers - layer_idx))
        else:
            lr = base_lr
        groups.append({"params": [p], "lr": lr, "weight_decay": decay})
    return groups

print("\n[2/2] Training 5-Fold Clean RoBERTa-large (Seed 2, LLRD, bf16)...")
model_name = "FacebookAI/roberta-large"
tokenizer = AutoTokenizer.from_pretrained(model_name)

class TextDS(Dataset):
    def __init__(self, texts, labels=None):
        self.enc = tokenizer(list(texts), truncation=True, max_length=128, padding=False)
        self.labels = list(labels) if labels is not None else None
    def __len__(self):
        return len(self.enc["input_ids"])
    def __getitem__(self, i):
        d = {"input_ids": self.enc["input_ids"][i], "attention_mask": self.enc["attention_mask"][i]}
        if self.labels is not None:
            d["labels"] = self.labels[i]
        return d

def collate(batch):
    ml = max(len(x["input_ids"]) for x in batch)
    pad_id = tokenizer.pad_token_id or 0
    ids, masks, labs = [], [], []
    for x in batch:
        pl = ml - len(x["input_ids"])
        ids.append(x["input_ids"] + [pad_id] * pl)
        masks.append(x["attention_mask"] + [0] * pl)
        if "labels" in x:
            labs.append(x["labels"])
    out = {"input_ids": torch.tensor(ids, dtype=torch.long), "attention_mask": torch.tensor(masks, dtype=torch.long)}
    if labs:
        out["labels"] = torch.tensor(labs, dtype=torch.long)
    return out

test_loader = DataLoader(TextDS(test_df["clean_text"]), batch_size=64, shuffle=False, collate_fn=collate)
class_weights = torch.tensor([(len(y_all) / (num_classes * counts[c])) ** 0.45 for c in range(num_classes)], dtype=torch.float32, device=device)

oof_rob_l2 = np.zeros((len(train_df), num_classes), dtype=np.float32)
test_rob_l2 = np.zeros((len(test_df), num_classes), dtype=np.float32)

for fold, (tr_idx, va_idx) in enumerate(skf.split(train_df, y_all), 1):
    t0 = time.time()
    seed_everything(SEED + fold * 19)
    tr_loader = DataLoader(TextDS(train_df.loc[tr_idx, "clean_text"], y_all[tr_idx]), batch_size=16, shuffle=True, collate_fn=collate)
    va_loader = DataLoader(TextDS(train_df.loc[va_idx, "clean_text"], y_all[va_idx]), batch_size=64, shuffle=False, collate_fn=collate)

    model = MeanMaxClassifier(model_name, num_classes).to(device)
    optimizer = torch.optim.AdamW(get_llrd_params(model))
    epochs, grad_accum = 4, 2
    total_steps = (len(tr_loader) // grad_accum) * epochs
    scheduler = get_cosine_schedule_with_warmup(optimizer, int(0.1 * total_steps), total_steps)
    criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.05)

    best_f1, best_va, best_state = -1.0, None, None
    for ep in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad()
        for step, b in enumerate(tr_loader, 1):
            with torch.amp.autocast("cuda", dtype=torch.bfloat16):
                logits = model(b["input_ids"].to(device), b["attention_mask"].to(device))
                loss = criterion(logits.float(), b["labels"].to(device)) / grad_accum
            loss.backward()
            if step % grad_accum == 0 or step == len(tr_loader):
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()

        model.eval()
        va_list = []
        with torch.inference_mode():
            for b in va_loader:
                with torch.amp.autocast("cuda", dtype=torch.bfloat16):
                    logits = model(b["input_ids"].to(device), b["attention_mask"].to(device))
                va_list.append(torch.softmax(logits.float(), dim=-1).cpu().numpy())
        va_probs = np.vstack(va_list)
        f1_ep = f1_score(y_all[va_idx], va_probs.argmax(axis=1), average="macro")
        if f1_ep > best_f1:
            best_f1, best_va = f1_ep, va_probs
            best_state = {k: v.cpu() for k, v in model.state_dict().items()}

    oof_rob_l2[va_idx] = best_va
    model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
    model.eval()
    te_list = []
    with torch.inference_mode():
        for b in test_loader:
            with torch.amp.autocast("cuda", dtype=torch.bfloat16):
                logits = model(b["input_ids"].to(device), b["attention_mask"].to(device))
            te_list.append(torch.softmax(logits.float(), dim=-1).cpu().numpy())
    test_rob_l2 += np.vstack(te_list) / 5.0
    print(f"  Fold {fold}/5 | Val Macro-F1: {best_f1:.4f} | {time.time() - t0:.1f}s")

print(f"--> Clean RoBERTa-large (Seed 2) 5-Fold OOF Macro-F1: {f1_score(y_all, oof_rob_l2.argmax(axis=1), average='macro'):.4f}")

# ==========================================
# 5. Strict 5-Fold Nested CV Class Multiplier Optimizer
# ==========================================
def fit_multipliers(probs, y_true, n_rounds=3):
    mult = np.ones(probs.shape[1], dtype=np.float32)
    best_score = f1_score(y_true, (probs * mult).argmax(axis=1), average="macro")
    grid = np.linspace(0.75, 1.35, 25)
    for _ in range(n_rounds):
        for c in range(probs.shape[1]):
            cur_best_m = mult[c]
            for g in grid:
                mult[c] = g
                sc = f1_score(y_true, (probs * mult).argmax(axis=1), average="macro")
                if sc > best_score + 1e-6:
                    best_score = sc
                    cur_best_m = g
            mult[c] = cur_best_m
    return mult

def nested_cv_multipliers(oof_probs, test_probs, y_true, damping=0.5):
    oof_full = np.zeros_like(oof_probs)
    oof_damp = np.zeros_like(oof_probs)
    fold_mults = []
    for tr_idx, va_idx in skf.split(oof_probs, y_true):
        m = fit_multipliers(oof_probs[tr_idx], y_true[tr_idx])
        fold_mults.append(m)
        m_damp = damping * m + (1.0 - damping) * 1.0
        oof_full[va_idx] = oof_probs[va_idx] * m
        oof_damp[va_idx] = oof_probs[va_idx] * m_damp
    avg_m = np.mean(fold_mults, axis=0)
    avg_m_damp = damping * avg_m + (1.0 - damping) * 1.0
    return (
        oof_full, test_probs * avg_m, avg_m,
        oof_damp, test_probs * avg_m_damp, avg_m_damp
    )

# ==========================================
# 6. Day 4 Final Ensembles (7-Family & 9-Family)
# ==========================================
# Day 3 Champion 7-Family Arith
w7 = np.array([0.28, 0.22, 0.14, 0.08, 0.10, 0.10, 0.08])
oofs7  = [oof_rob_l1, oof_deb_l, oof_rob_b, oof_deb_b, oof_bge, oof_e5, oof_lex]
tests7 = [test_rob_l1, test_deb_l, test_rob_b, test_deb_b, test_bge, test_e5, test_lex]
oof_7f_raw  = sum(w * o for w, o in zip(w7, oofs7))
test_7f_raw = sum(w * t for w, t in zip(w7, tests7))
_, _, _, oof_7f_damp, test_7f_damp, m7_damp = nested_cv_multipliers(oof_7f_raw, test_7f_raw, y_all, damping=0.5)

# Day 4 Grand 9-Family Arith
# [0] RoB-L1, [1] RoB-L2, [2] DeB-L, [3] RoB-B, [4] DeB-B, [5] BGE, [6] E5, [7] MPNet, [8] Lex
w9 = np.array([0.20, 0.20, 0.18, 0.10, 0.06, 0.07, 0.07, 0.06, 0.06])
oofs9  = [oof_rob_l1, oof_rob_l2, oof_deb_l, oof_rob_b, oof_deb_b, oof_bge, oof_e5, oof_mpnet, oof_lex]
tests9 = [test_rob_l1, test_rob_l2, test_deb_l, test_rob_b, test_deb_b, test_bge, test_e5, test_mpnet, test_lex]
oof_9f_raw  = sum(w * o for w, o in zip(w9, oofs9))
test_9f_raw = sum(w * t for w, t in zip(w9, tests9))
oof_9f_full, test_9f_full, m9_full, oof_9f_damp, test_9f_damp, m9_damp = nested_cv_multipliers(oof_9f_raw, test_9f_raw, y_all, damping=0.5)

# 9-Family L2 Meta-Stacker (C=0.10)
X_tr_9 = np.hstack(oofs9)
X_te_9 = np.hstack(tests9)
oof_9f_meta = np.zeros((len(train_df), num_classes), dtype=np.float32)
test_9f_meta = np.zeros((len(test_df), num_classes), dtype=np.float32)
for fold, (tr_idx, va_idx) in enumerate(skf.split(X_tr_9, y_all)):
    clf = LogisticRegression(C=0.10, max_iter=1000, random_state=42 + fold)
    clf.fit(X_tr_9[tr_idx], y_all[tr_idx])
    oof_9f_meta[va_idx] = clf.predict_proba(X_tr_9[va_idx])
    test_9f_meta += clf.predict_proba(X_te_9) / 5.0

print("\n" + "=" * 78)
print("DAY 4 FINAL 9-FAMILY ENSEMBLE SUMMARY (100% COMPLIANT)")
print("=" * 78)
print(f"  [Ref] Day 3 7-Family Raw Arith          : {f1_score(y_all, oof_7f_raw.argmax(axis=1), average='macro'):.4f}")
print(f"  1. 7-Family Arith + Damped Mult (0.5x)  : {f1_score(y_all, oof_7f_damp.argmax(axis=1), average='macro'):.4f} | Mult: {np.round(m7_damp, 3)}")
print(f"  2. 9-Family Arith Raw Argmax            : {f1_score(y_all, oof_9f_raw.argmax(axis=1), average='macro'):.4f}")
print(f"  3. 9-Family Arith + Damped Mult (0.5x)  : {f1_score(y_all, oof_9f_damp.argmax(axis=1), average='macro'):.4f} | Mult: {np.round(m9_damp, 3)}")
print(f"  4. 9-Family Arith + Full Nested Mult    : {f1_score(y_all, oof_9f_full.argmax(axis=1), average='macro'):.4f} | Mult: {np.round(m9_full, 3)}")
print(f"  5. 9-Family L2 Meta-Stacker (C=0.10)    : {f1_score(y_all, oof_9f_meta.argmax(axis=1), average='macro'):.4f}")
print("=" * 78)

subs_day4 = {
    "sub_day4_1_7fam_arith_damped_mult.csv": test_7f_damp,
    "sub_day4_2_9fam_arith_raw.csv": test_9f_raw,
    "sub_day4_3_9fam_arith_damped_mult.csv": test_9f_damp,
    "sub_day4_4_9fam_arith_full_mult.csv": test_9f_full,
    "sub_day4_5_9fam_meta_l2.csv": test_9f_meta,
}
for fname, probs in subs_day4.items():
    pd.DataFrame({"id": test_df[sub_id_col], "emotion": le.inverse_transform(probs.argmax(axis=1))}).to_csv(fname, index=False)
    files.download(fname)

np.savez(
    "clean_day4_all_probs.npz",
    oof_rob_l1=oof_rob_l1, test_rob_l1=test_rob_l1,
    oof_rob_l2=oof_rob_l2, test_rob_l2=test_rob_l2,
    oof_deb_l=oof_deb_l,   test_deb_l=test_deb_l,
    oof_rob_b=oof_rob_b,   test_rob_b=test_rob_b,
    oof_deb_b=oof_deb_b,   test_deb_b=test_deb_b,
    oof_bge=oof_bge,       test_bge=test_bge,
    oof_e5=oof_e5,         test_e5=test_e5,
    oof_mpnet=oof_mpnet,   test_mpnet=test_mpnet,
    oof_lex=oof_lex,       test_lex=test_lex,
)
files.download("clean_day4_all_probs.npz")

Using device: cuda
Loaded Day 3 clean probabilities successfully!

[1/2] Extracting Clean all-mpnet-base-v2 Embeddings (zero emotion supervision)...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

... (39 outputs left out)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>